# PCCP - Full Publication Figures with v8 Settings
## Physics-Constrained Conformal Prediction for RUL Estimation

**v8 Settings Applied:**
- Pure MSE loss (no physics loss during training)
- 5 UQ methods: CP, MC Dropout, Deep Ensemble, QR, CQR
- MC: train_dropout=0.2, inference_dropout=0.3, 100 samples
- Ensemble: 10 models with bootstrap
- All 4 C-MAPSS datasets (FD001-FD004)

**Figures from Full Figures notebook, redrawn with v8 pipeline.**

In [ ]:
%pip install torch numpy pandas matplotlib scikit-learn tqdm scipy seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle, Rectangle
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy import stats
from tqdm import tqdm
from datetime import datetime
import os, warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# ============================================================
# V8 CONFIGURATION
# ============================================================
DATA_PATH = os.environ.get('CMAPSS_DIR', 'data/CMAPSSData')

MAX_RUL = 125
WINDOW_SIZE = 30
HIDDEN_DIMS = [128, 64, 32]
DROPOUT = 0.2
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
N_EPOCHS = 200
PATIENCE = 30

# MC Dropout settings (v8)
MC_TRAIN_DROPOUT = 0.2
MC_INFERENCE_DROPOUT = 0.3
MC_SAMPLES = 100

# Ensemble settings (v8)
N_ENSEMBLE = 10

# NOTE: No LAMBDA_NN - v8 uses pure MSE loss (NOT physics_loss)
print("✓ v8 Configuration loaded")
print("  - Training loss: Pure MSE (no physics penalty)")
print(f"  - MC Dropout: train={MC_TRAIN_DROPOUT}, inference={MC_INFERENCE_DROPOUT}, samples={MC_SAMPLES}")
print(f"  - Ensemble: {N_ENSEMBLE} models with bootstrap")


In [ ]:
# ============================================================
# PUBLICATION FIGURE SETTINGS (600 DPI, Nature-style)
# ============================================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'legend.fontsize': 10,
    'legend.framealpha': 0.9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.3,
    'lines.linewidth': 1.5,
    'lines.markersize': 6,
})

COLORS = {
    'cp': '#2E86AB',
    'pccp': '#28A745',
    'infeasible': '#DC3545',
    'warning': '#FFC107',
    'neutral': '#6C757D',
    'dark': '#1D3557',
    'light_blue': '#A8DADC',
    'light_green': '#90EE90',
    'light_red': '#FFCCCB',
    'purple': '#7B2CBF',
    'orange': '#F18F01',
    'teal': '#17A2B8',
}

# Method colors (v8 style)
METHOD_COLORS = {
    'CP': '#0072B2',
    'MC Dropout': '#E69F00',
    'Ensemble': '#009E73',
    'QR': '#CC79A7',
    'CQR': '#D55E00',
}

sns.set_palette([COLORS['cp'], COLORS['pccp'], COLORS['infeasible'], COLORS['purple']])
print("✓ Publication figure settings configured (600 DPI)")


In [ ]:
# ============================================================
# DATA LOADING (v8 - supports all datasets)
# ============================================================
COLUMNS = ['unit_id', 'time'] + [f'op_{i}' for i in range(1, 4)] + [f's_{i}' for i in range(1, 22)]
DROP_SENSORS = ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']

def load_dataset(dataset_name):
    train_path = os.path.join(DATA_PATH, f'train_{dataset_name}.txt')
    test_path = os.path.join(DATA_PATH, f'test_{dataset_name}.txt')
    rul_path = os.path.join(DATA_PATH, f'RUL_{dataset_name}.txt')
    
    train_df = pd.read_csv(train_path, sep=' ', header=None)
    train_df.drop(train_df.columns[[-1, -2]], axis=1, inplace=True)
    train_df.columns = COLUMNS
    
    test_df = pd.read_csv(test_path, sep=' ', header=None)
    test_df.drop(test_df.columns[[-1, -2]], axis=1, inplace=True)
    test_df.columns = COLUMNS
    
    test_rul = pd.read_csv(rul_path, sep=' ', header=None)
    test_rul.drop(test_rul.columns[[-1]], axis=1, inplace=True)
    test_rul = test_rul.values.flatten()
    
    # Store raw for visualization
    train_df_raw = train_df.copy()
    
    max_time_train = train_df.groupby('unit_id')['time'].max().reset_index()
    max_time_train.columns = ['unit_id', 'max_time']
    train_df = train_df.merge(max_time_train, on='unit_id')
    train_df['RUL'] = (train_df['max_time'] - train_df['time']).clip(upper=MAX_RUL)
    train_df.drop('max_time', axis=1, inplace=True)
    
    max_time_test = test_df.groupby('unit_id')['time'].max().reset_index()
    max_time_test.columns = ['unit_id', 'max_time']
    max_time_test['final_rul'] = test_rul
    test_df = test_df.merge(max_time_test, on='unit_id')
    test_df['RUL'] = (test_df['final_rul'] + test_df['max_time'] - test_df['time']).clip(upper=MAX_RUL)
    test_df.drop(['max_time', 'final_rul'], axis=1, inplace=True)
    
    train_df.drop(columns=DROP_SENSORS, inplace=True, errors='ignore')
    test_df.drop(columns=DROP_SENSORS, inplace=True, errors='ignore')
    feature_cols = [c for c in train_df.columns if c not in ['unit_id', 'time', 'RUL']]
    
    scaler = StandardScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols] = scaler.transform(test_df[feature_cols])
    
    return train_df, test_df, feature_cols, train_df_raw

def create_sequences(df, window_size, feature_cols):
    X_list, y_list, t_list, unit_list = [], [], [], []
    for unit_id in sorted(df['unit_id'].unique()):
        unit_df = df[df['unit_id'] == unit_id].sort_values('time')
        features = unit_df[feature_cols].values
        rul = unit_df['RUL'].values
        time = unit_df['time'].values
        if len(features) < window_size:
            continue
        for i in range(len(features) - window_size + 1):
            X_list.append(features[i:i + window_size].flatten())
            y_list.append(rul[i + window_size - 1])
            t_list.append(time[i + window_size - 1])
            unit_list.append(unit_id)
    return np.array(X_list), np.array(y_list), np.array(t_list), np.array(unit_list)

def prepare_data(dataset_name):
    train_df, test_df, feature_cols, train_df_raw = load_dataset(dataset_name)
    X_train_all, y_train_all, t_train_all, units_train_all = create_sequences(train_df, WINDOW_SIZE, feature_cols)
    X_test_all, y_test_all, t_test_all, units_test_all = create_sequences(test_df, WINDOW_SIZE, feature_cols)
    
    unique_train_units = np.unique(units_train_all)
    rng = np.random.RandomState(SEED)
    shuffled = unique_train_units.copy()
    rng.shuffle(shuffled)
    n_units = len(shuffled)
    train_unit_ids = shuffled[:int(0.7 * n_units)]
    val_unit_ids = shuffled[int(0.7 * n_units):int(0.85 * n_units)]
    cal_unit_ids = shuffled[int(0.85 * n_units):]
    
    train_mask = np.isin(units_train_all, train_unit_ids)
    val_mask = np.isin(units_train_all, val_unit_ids)
    cal_mask = np.isin(units_train_all, cal_unit_ids)
    
    # Visualization data for trajectory plots
    viz_units_data = []
    for unit_id in val_unit_ids[:5]:  # First 5 val units
        mask = units_train_all == unit_id
        if mask.sum() > 0:
            viz_units_data.append({
                'unit_id': unit_id,
                'X': X_train_all[mask].astype(np.float32),
                'y': y_train_all[mask],
                't': t_train_all[mask]
            })
    
    return {
        'X_train': X_train_all[train_mask].astype(np.float32),
        'y_train': y_train_all[train_mask].astype(np.float32),
        'X_val': X_train_all[val_mask].astype(np.float32),
        'y_val': y_train_all[val_mask].astype(np.float32),
        'X_cal': X_train_all[cal_mask].astype(np.float32),
        'y_cal': y_train_all[cal_mask].astype(np.float32),
        'X_test': X_test_all.astype(np.float32),
        'y_test': y_test_all.astype(np.float32),
        't_test': t_test_all,
        'units_test': units_test_all,
        'n_features': X_train_all.shape[1],
        'feature_cols': feature_cols,
        'train_df_raw': train_df_raw,
        'viz_units_data': viz_units_data,
    }

print("✓ Data loading functions ready")


In [ ]:
# ============================================================
# MODELS (v8 - all UQ methods)
# ============================================================

class RULPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev_dim, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

class MCDropoutPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], train_dropout=0.2):
        super().__init__()
        self.hidden_dims = hidden_dims
        self.train_dropout = train_dropout
        self.linears = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        prev_dim = input_dim
        for h in hidden_dims:
            self.linears.append(nn.Linear(prev_dim, h))
            self.dropouts.append(nn.Dropout(train_dropout))
            prev_dim = h
        self.output = nn.Linear(prev_dim, 1)
    
    def forward(self, x):
        for linear, dropout in zip(self.linears, self.dropouts):
            x = dropout(torch.relu(linear(x)))
        return self.output(x).squeeze(-1)
    
    def set_inference_dropout(self, p):
        for dropout in self.dropouts:
            dropout.p = p
    
    def predict_with_uncertainty(self, X, n_samples=100, inference_dropout=0.3):
        self.set_inference_dropout(inference_dropout)
        self.train()
        X_tensor = torch.FloatTensor(X).to(device)
        predictions = []
        with torch.no_grad():
            for _ in range(n_samples):
                pred = self.forward(X_tensor).cpu().numpy()
                predictions.append(pred)
        self.set_inference_dropout(self.train_dropout)
        self.eval()
        return np.array(predictions)

class QuantilePredictor(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev_dim, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 3))  # lower, median, upper
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

print("✓ All model classes defined (RULPredictor, MCDropoutPredictor, QuantilePredictor)")


In [ ]:
# ============================================================
# TRAINING (v8 - pure MSE loss, WITH history for figures)
# ============================================================

def train_model_with_history(data, model_class=RULPredictor, seed_offset=0, dropout=0.2, verbose=True):
    """v8 training with pure MSE loss + history tracking for training curve figures"""
    X_train, y_train = data['X_train'], data['y_train']
    X_val, y_val = data['X_val'], data['y_val']
    n_features = data['n_features']
    
    torch.manual_seed(SEED + seed_offset)
    np.random.seed(SEED + seed_offset)
    
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
                            batch_size=BATCH_SIZE)
    
    if model_class == MCDropoutPredictor:
        model = model_class(n_features, HIDDEN_DIMS, train_dropout=dropout).to(device)
    else:
        model = model_class(n_features, HIDDEN_DIMS, dropout).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    
    best_val_loss = float('inf')
    best_val_rmse = float('inf')
    patience_counter = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_rmse': [], 'lr': []}
    
    pbar = tqdm(range(N_EPOCHS), desc="Training") if verbose else range(N_EPOCHS)
    for epoch in pbar:
        # Train - v8 uses PURE MSE (no physics_loss!)
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = nn.functional.mse_loss(model(X_batch), y_batch)  # PURE MSE
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        # Validate
        model.eval()
        val_losses = []
        val_preds, val_targets = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                val_losses.append(nn.functional.mse_loss(pred, y_batch).item())
                val_preds.extend(pred.cpu().numpy())
                val_targets.extend(y_batch.cpu().numpy())
        val_loss = np.mean(val_losses)
        val_rmse = np.sqrt(np.mean((np.array(val_preds) - np.array(val_targets))**2))
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)
        history['lr'].append(scheduler.get_last_lr()[0])
        scheduler.step()
        
        if verbose:
            pbar.set_postfix({'loss': f'{val_loss:.4f}', 'rmse': f'{val_rmse:.2f}'})
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                if verbose:
                    print(f"\n   Early stopping at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    if verbose:
        print(f"   Best val RMSE: {best_val_rmse:.2f}")
    return model, history

def train_model(data, model_class=RULPredictor, seed_offset=0, dropout=0.2, verbose=False):
    """v8 training without history (for ensemble/quick training)"""
    model, _ = train_model_with_history(data, model_class, seed_offset, dropout, verbose)
    return model

def train_ensemble_bootstrap(data, n_models=10):
    models = []
    X_train, y_train = data['X_train'], data['y_train']
    n_samples = len(X_train)
    for i in range(n_models):
        print(f"         Training ensemble model {i+1}/{n_models}...", end='\r')
        rng = np.random.RandomState(SEED + i * 1000)
        indices = rng.choice(n_samples, size=n_samples, replace=True)
        data_boot = {
            'X_train': X_train[indices], 'y_train': y_train[indices],
            'X_val': data['X_val'], 'y_val': data['y_val'],
            'n_features': data['n_features']
        }
        model = train_model(data_boot, RULPredictor, seed_offset=i*1000, dropout=DROPOUT, verbose=False)
        models.append(model)
    print(f"         ✓ Trained {n_models} ensemble models" + " "*20)
    return models

def train_quantile_model(data, alpha=0.1):
    X_train, y_train = data['X_train'], data['y_train']
    X_val, y_val = data['X_val'], data['y_val']
    n_features = data['n_features']
    
    torch.manual_seed(SEED + 300)
    model = QuantilePredictor(n_features, HIDDEN_DIMS, DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    
    quantiles = [alpha/2, 0.5, 1 - alpha/2]
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
                            batch_size=BATCH_SIZE)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    for epoch in range(N_EPOCHS):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = 0
            for k, q in enumerate(quantiles):
                errors = y_batch - out[:, k]
                loss += torch.mean(torch.max(q * errors, (q - 1) * errors))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch)
                for k, q in enumerate(quantiles):
                    errors = y_batch - out[:, k]
                    val_loss += torch.mean(torch.max(q * errors, (q - 1) * errors)).item()
        scheduler.step()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break
    
    model.load_state_dict(best_state)
    return model

print("✓ All training functions ready (pure MSE loss)")


In [ ]:
# ============================================================
# PREDICTION FUNCTIONS (v8)
# ============================================================

@torch.no_grad()
def get_predictions(model, X):
    model.eval()
    return model(torch.FloatTensor(X).to(device)).cpu().numpy()

def calibrate_cp(model, X_cal, y_cal, alpha):
    pred = get_predictions(model, X_cal)
    scores = np.abs(y_cal - pred)
    n = len(scores)
    q_level = min(np.ceil((1 - alpha) * (n + 1)) / n, 1.0)
    return np.quantile(scores, q_level), scores

def predict_cp(model, X, q_hat):
    pred = get_predictions(model, X)
    return {'pred': pred, 'lower': pred - q_hat, 'upper': pred + q_hat}

def predict_mc_dropout(model, X, alpha=0.1, n_samples=100, inference_dropout=0.3):
    predictions = model.predict_with_uncertainty(X, n_samples, inference_dropout)
    mean = predictions.mean(axis=0)
    std = predictions.std(axis=0)
    z = stats.norm.ppf(1 - alpha/2)
    return {'pred': mean, 'lower': mean - z * std, 'upper': mean + z * std, 'std': std}

def get_ensemble_predictions(models, X):
    X_tensor = torch.FloatTensor(X).to(device)
    predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            predictions.append(model(X_tensor).cpu().numpy())
    return np.array(predictions)

def calibrate_ensemble(models, X_cal, y_cal, alpha):
    predictions = get_ensemble_predictions(models, X_cal)
    mean = predictions.mean(axis=0)
    scores = np.abs(y_cal - mean)
    n = len(scores)
    q_level = min(np.ceil((1 - alpha) * (n + 1)) / n, 1.0)
    return np.quantile(scores, q_level)

def predict_ensemble_calibrated(models, X, q_hat):
    predictions = get_ensemble_predictions(models, X)
    mean = predictions.mean(axis=0)
    std = predictions.std(axis=0)
    return {'pred': mean, 'lower': mean - q_hat, 'upper': mean + q_hat, 'std': std}

def predict_quantile(model, X):
    model.eval()
    with torch.no_grad():
        out = model(torch.FloatTensor(X).to(device)).cpu().numpy()
    return {'pred': out[:, 1], 'lower': out[:, 0], 'upper': out[:, 2]}

def calibrate_cqr(model, X_cal, y_cal, alpha):
    qr_results = predict_quantile(model, X_cal)
    scores = np.maximum(qr_results['lower'] - y_cal, y_cal - qr_results['upper'])
    n = len(scores)
    q_level = min(np.ceil((1 - alpha) * (n + 1)) / n, 1.0)
    return np.quantile(scores, q_level)

def predict_cqr(model, X, q_hat):
    qr_results = predict_quantile(model, X)
    return {'pred': qr_results['pred'], 
            'lower': qr_results['lower'] - q_hat, 
            'upper': qr_results['upper'] + q_hat}

print("✓ All prediction functions ready")


In [ ]:
# ============================================================
# METRICS (v8)
# ============================================================

def compute_all_metrics(y_true, results):
    pred = results['pred']
    lower_raw = results['lower']
    upper_raw = results['upper']
    
    # BEFORE physics constraint
    covered_before = (y_true >= lower_raw) & (y_true <= upper_raw)
    picp_before = 100 * covered_before.mean()
    mpiw_before = np.mean(upper_raw - lower_raw)
    neg_before = 100 * np.mean(lower_raw < 0)
    over_before = 100 * np.mean(upper_raw > MAX_RUL)
    phys_before = max(0, 100 - neg_before - over_before)
    
    # AFTER physics constraint
    lower_pc = np.maximum(lower_raw, 0)
    upper_pc = np.minimum(upper_raw, MAX_RUL)
    covered_after = (y_true >= lower_pc) & (y_true <= upper_pc)
    picp_after = 100 * covered_after.mean()
    mpiw_after = np.mean(upper_pc - lower_pc)
    
    rmse = np.sqrt(np.mean((y_true - pred) ** 2))
    mpiw_reduction = 100 * (mpiw_before - mpiw_after) / mpiw_before if mpiw_before > 0 else 0
    
    return {
        'RMSE': rmse,
        'before': {'PICP': picp_before, 'MPIW': mpiw_before, 'Neg%': neg_before, 'Over%': over_before, 'Phys%': phys_before},
        'after': {'PICP': picp_after, 'MPIW': mpiw_after, 'Neg%': 0.0, 'Over%': 0.0, 'Phys%': 100.0},
        'MPIW_reduction_%': mpiw_reduction,
        'PICP_change': picp_after - picp_before,
    }

def compute_critical_metrics(y_true, results, threshold):
    mask = y_true < threshold
    if mask.sum() == 0:
        return None
    return compute_all_metrics(y_true[mask], {k: v[mask] for k, v in results.items()})

print("✓ Metrics functions ready")


In [ ]:
# ============================================================
# RUN EXPERIMENT (v8 - all 5 methods, FD001 for figures)
# ============================================================

ALPHA = 0.1  # 90% coverage
LAMBDA_NN = 0.0  # v8 uses pure MSE, but some figure cells reference this

# Load FD001 data
print("📂 Loading FD001 data...")
data = prepare_data('FD001')
X_test, y_test = data['X_test'], data['y_test']
X_cal, y_cal = data['X_cal'], data['y_cal']
viz_units_data = data['viz_units_data']
train_df_raw = data['train_df_raw']
n_features = data['n_features']

print(f"   Train: {len(data['X_train'])}, Val: {len(data['X_val'])}")
print(f"   Cal: {len(X_cal)}, Test: {len(X_test)}")
print(f"   Features: {n_features}, Viz engines: {len(viz_units_data)}")

all_raw = {}
all_metrics_v8 = {}

# 1. CP (with history for training curves)
print("\n[1/5] Training CP model...")
model_cp, history = train_model_with_history(data, RULPredictor, seed_offset=0, dropout=DROPOUT)
q_hat, cal_scores = calibrate_cp(model_cp, X_cal, y_cal, ALPHA)
print(f"       q_hat = {q_hat:.2f}")
all_raw['CP'] = predict_cp(model_cp, X_test, q_hat)
all_metrics_v8['CP'] = compute_all_metrics(y_test, all_raw['CP'])
print(f"       RMSE={all_metrics_v8['CP']['RMSE']:.2f}, PICP={all_metrics_v8['CP']['before']['PICP']:.1f}%")

# Also compute CP results at 95% for some figures
q_95, cal_scores_95 = calibrate_cp(model_cp, X_cal, y_cal, alpha=0.05)

# ============================================================
# BACKWARD-COMPATIBLE ALIASES (for Full Figures code)
# ============================================================
model = model_cp
q_90 = q_hat
cal_scores_90 = cal_scores

# CP/PCCP raw results with lower_raw/upper_raw keys
results_cp_90 = {
    'pred': all_raw['CP']['pred'].copy(),
    'lower': all_raw['CP']['lower'].copy(),
    'upper': all_raw['CP']['upper'].copy(),
    'lower_raw': all_raw['CP']['lower'].copy(),
    'upper_raw': all_raw['CP']['upper'].copy(),
}

results_pccp_90 = {
    'pred': all_raw['CP']['pred'].copy(),
    'lower': np.maximum(all_raw['CP']['lower'], 0),
    'upper': np.minimum(all_raw['CP']['upper'], MAX_RUL),
    'lower_raw': all_raw['CP']['lower'].copy(),
    'upper_raw': all_raw['CP']['upper'].copy(),
}

results_cp_95 = predict_cp(model_cp, X_test, q_95)
results_cp_95['lower_raw'] = results_cp_95['lower'].copy()
results_cp_95['upper_raw'] = results_cp_95['upper'].copy()

results_pccp_95 = {
    'pred': results_cp_95['pred'].copy(),
    'lower': np.maximum(results_cp_95['lower'], 0),
    'upper': np.minimum(results_cp_95['upper'], MAX_RUL),
    'lower_raw': results_cp_95['lower'].copy(),
    'upper_raw': results_cp_95['upper'].copy(),
}

# OLD-FORMAT FLAT METRICS (figure cells access metrics['PICP'] not metrics['before']['PICP'])
def to_flat_metrics(y_true, results, method_name):
    """Create old-format flat metrics dict for figure compatibility"""
    pred = results['pred']
    lower = results['lower']
    upper = results['upper']
    lower_raw = results.get('lower_raw', lower)
    
    rmse = np.sqrt(np.mean((y_true - pred) ** 2))
    mae = np.mean(np.abs(y_true - pred))
    covered = (y_true >= lower) & (y_true <= upper)
    picp = 100 * np.mean(covered)
    mpiw = np.mean(upper - lower)
    neg_pct = 100 * np.mean(lower_raw < 0)
    phys_pct = 100.0 if 'PCCP' in method_name else max(0, 100 - neg_pct)
    
    return {'RMSE': rmse, 'MAE': mae, 'PICP': picp, 'MPIW': mpiw, 'Neg%': neg_pct, 'Phys%': phys_pct}

metrics_cp_90 = to_flat_metrics(y_test, results_cp_90, 'CP')
metrics_pccp_90 = to_flat_metrics(y_test, results_pccp_90, 'PCCP')
metrics_cp_95 = to_flat_metrics(y_test, results_cp_95, 'CP')
metrics_pccp_95 = to_flat_metrics(y_test, results_pccp_95, 'PCCP')
mpiw_impr_90 = 100 * (metrics_cp_90['MPIW'] - metrics_pccp_90['MPIW']) / metrics_cp_90['MPIW']

print(f"\n   CP  (90%): PICP={metrics_cp_90['PICP']:.1f}%, MPIW={metrics_cp_90['MPIW']:.2f}, Neg%={metrics_cp_90['Neg%']:.1f}%")
print(f"   PCCP(90%): PICP={metrics_pccp_90['PICP']:.1f}%, MPIW={metrics_pccp_90['MPIW']:.2f}, Phys%=100.0%")
print(f"   MPIW Improvement: {mpiw_impr_90:.1f}%")

# ============================================================
# REMAINING UQ METHODS
# ============================================================

# 2. MC Dropout
print("\n[2/5] Training MC Dropout...")
model_mc, _ = train_model_with_history(data, MCDropoutPredictor, seed_offset=0, dropout=MC_TRAIN_DROPOUT)
all_raw['MC Dropout'] = predict_mc_dropout(model_mc, X_test, ALPHA, MC_SAMPLES, MC_INFERENCE_DROPOUT)
all_metrics_v8['MC Dropout'] = compute_all_metrics(y_test, all_raw['MC Dropout'])
print(f"       RMSE={all_metrics_v8['MC Dropout']['RMSE']:.2f}, PICP={all_metrics_v8['MC Dropout']['before']['PICP']:.1f}%")

# 3. Deep Ensemble (Calibrated)
print("\n[3/5] Training Deep Ensemble (Calibrated)...")
models_ens = train_ensemble_bootstrap(data, N_ENSEMBLE)
q_ens = calibrate_ensemble(models_ens, X_cal, y_cal, ALPHA)
print(f"       q_ens = {q_ens:.2f}")
all_raw['Ensemble'] = predict_ensemble_calibrated(models_ens, X_test, q_ens)
all_metrics_v8['Ensemble'] = compute_all_metrics(y_test, all_raw['Ensemble'])
print(f"       RMSE={all_metrics_v8['Ensemble']['RMSE']:.2f}, PICP={all_metrics_v8['Ensemble']['before']['PICP']:.1f}%")

# 4. Quantile Regression
print("\n[4/5] Training Quantile Regression...")
model_qr = train_quantile_model(data, ALPHA)
all_raw['QR'] = predict_quantile(model_qr, X_test)
all_metrics_v8['QR'] = compute_all_metrics(y_test, all_raw['QR'])
print(f"       RMSE={all_metrics_v8['QR']['RMSE']:.2f}, PICP={all_metrics_v8['QR']['before']['PICP']:.1f}%")

# 5. CQR
print("\n[5/5] Computing CQR...")
q_cqr = calibrate_cqr(model_qr, X_cal, y_cal, ALPHA)
print(f"       q_cqr = {q_cqr:.2f}")
all_raw['CQR'] = predict_cqr(model_qr, X_test, q_cqr)
all_metrics_v8['CQR'] = compute_all_metrics(y_test, all_raw['CQR'])
print(f"       PICP={all_metrics_v8['CQR']['before']['PICP']:.1f}%")

# Critical region metrics (v8 format)
all_critical = {}
for thresh in [50, 30, 15]:
    key = f'RUL<{thresh}'
    all_critical[key] = {}
    for method in all_raw:
        cm = compute_critical_metrics(y_test, all_raw[method], thresh)
        if cm:
            all_critical[key][method] = cm

# Print summary
print("\n" + "="*70)
print("SUMMARY - FD001 (v8 settings)")
print("="*70)
METHODS = ['CP', 'MC Dropout', 'Ensemble', 'QR', 'CQR']
print(f"{'Method':<15} {'RMSE':>6} {'PICP_B':>7} {'PICP_A':>7} {'MPIW_B':>7} {'MPIW_A':>7} {'Δ%':>6} {'Neg%':>5}")
for m in METHODS:
    r = all_metrics_v8[m]
    print(f"{m:<15} {r['RMSE']:6.2f} {r['before']['PICP']:6.1f}% {r['after']['PICP']:6.1f}% "
          f"{r['before']['MPIW']:7.2f} {r['after']['MPIW']:7.2f} {r['MPIW_reduction_%']:5.1f}% {r['before']['Neg%']:4.1f}%")
print("\n✓ All experiments complete!")
print("\n✓ Backward-compatible variables created:")
print("  model, q_90, q_95, cal_scores_90, cal_scores_95")
print("  results_cp_90, results_pccp_90, results_cp_95, results_pccp_95")
print("  metrics_cp_90, metrics_pccp_90, metrics_cp_95, metrics_pccp_95 (FLAT format)")
print(f"  mpiw_impr_90={mpiw_impr_90:.1f}%, n_features={n_features}, LAMBDA_NN={LAMBDA_NN}")


# ============================================================
# ADDITIONAL BACKWARD-COMPATIBLE ALIASES
# ============================================================
# Figure cells reference these directly (from old flat namespace)

# Data arrays (old code used flat variables, v8 stores in data dict)
y_train = data['y_train']
y_val = data['y_val']
X_train = data['X_train']
X_val = data['X_val']
# X_cal and y_cal already defined above
# X_test and y_test already defined above
# train_df_raw already defined above

# DataLoaders (for cells that reference them)
train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
                        batch_size=BATCH_SIZE)

print("\n✓ All backward-compatible aliases created")
print(f"  y_train: {y_train.shape}, y_val: {y_val.shape}")
print(f"  X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"  X_cal: {X_cal.shape}, X_test: {X_test.shape}")


# ============================================================
# OLD FUNCTION ALIASES (for figure cells compatibility)
# ============================================================

def calibrate(model, X_cal, y_cal, alpha):
    """Old-format calibrate (returns q_hat, scores)"""
    return calibrate_cp(model, X_cal, y_cal, alpha)

@torch.no_grad()
def predict_intervals(model, X, q_hat, apply_physics=False):
    """Old-format predict_intervals with apply_physics flag"""
    model.eval()
    pred = model(torch.FloatTensor(X).to(device)).cpu().numpy()
    lower_raw = pred - q_hat
    upper_raw = pred + q_hat
    
    if apply_physics:
        lower = np.maximum(lower_raw, 0)
        upper = np.minimum(upper_raw, MAX_RUL)
    else:
        lower = lower_raw
        upper = upper_raw
    
    return {'pred': pred, 'lower': lower, 'upper': upper, 'lower_raw': lower_raw, 'upper_raw': upper_raw}

def compute_metrics(y_true, results, method_name=''):
    """Old-format compute_metrics (returns flat dict)"""
    pred, lower, upper = results['pred'], results['lower'], results['upper']
    lower_raw = results.get('lower_raw', lower)
    
    rmse = np.sqrt(np.mean((y_true - pred) ** 2))
    mae = np.mean(np.abs(y_true - pred))
    covered = (y_true >= lower) & (y_true <= upper)
    picp = 100 * np.mean(covered)
    mpiw = np.mean(upper - lower)
    neg_pct = 100 * np.mean(lower_raw < 0)
    phys_pct = 100.0 if 'PCCP' in method_name else max(0, 100 - neg_pct)
    
    return {'RMSE': rmse, 'MAE': mae, 'PICP': picp, 'MPIW': mpiw, 'Neg%': neg_pct, 'Phys%': phys_pct}

def physics_loss(pred, target, lambda_nn=0.0):
    """Old physics_loss (v8 uses pure MSE so lambda_nn=0.0 by default)"""
    mse = nn.functional.mse_loss(pred, target)
    nn_penalty = torch.mean(torch.relu(-pred) ** 2)
    return mse + lambda_nn * nn_penalty

print("\n✓ Old-format function aliases created:")
print("  calibrate(), predict_intervals(), compute_metrics(), physics_loss()")


---
## Figures
### Figure 1: PCCP Framework Schematic

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle, Rectangle, ConnectionPatch
from matplotlib.lines import Line2D

# ============================================================
# COLOR PALETTE
# ============================================================
COLORS_PRO = {
    'primary': '#2E86AB',      
    'secondary': '#28A745',    
    'accent': '#F18F01',       
    'purple': '#9C27B0',
    
    'bg_blue': '#E3F2FD',
    'bg_green': '#E8F5E9',
    'bg_orange': '#FFF3E0',
    'bg_purple': '#F3E5F5',
    'bg_gray': '#FAFAFA',
    
    'dark': '#212121',
    'gray': '#757575',
    'light_gray': '#E0E0E0',
    'white': '#FFFFFF'
}

# ============================================================
# CREATE FIGURE
# ============================================================
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_facecolor('white')

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def draw_box(ax, x, y, w, h, facecolor, edgecolor, linewidth=2):
    """Draw rounded box"""
    box = FancyBboxPatch((x, y), w, h, 
                          boxstyle="round,pad=0.02,rounding_size=0.12",
                          facecolor=facecolor, edgecolor=edgecolor, 
                          linewidth=linewidth)
    ax.add_patch(box)
    return box

def draw_arrow_simple(ax, x1, y1, x2, y2, color, lw=2.5):
    """Draw simple straight arrow"""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                               connectionstyle='arc3,rad=0'))

def draw_arrow_curved(ax, x1, y1, x2, y2, color, rad=0.2, lw=2.5):
    """Draw curved arrow"""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                               connectionstyle=f'arc3,rad={rad}'))

# ============================================================
# STAGE HEADERS WITH NUMBERS
# ============================================================
stages = [
    (1.5, 'DATA', COLORS_PRO['primary']),
    (4.5, 'TRAINING', COLORS_PRO['purple']),
    (7.8, 'CALIBRATION', COLORS_PRO['secondary']),
    (11.2, 'PROJECTION', COLORS_PRO['accent']),
    (14.2, 'OUTPUT', COLORS_PRO['secondary'])
]

for i, (x, label, color) in enumerate(stages):
    # Number circle
    circle = Circle((x, 8.3), 0.22, facecolor=color, edgecolor='white', linewidth=2, zorder=10)
    ax.add_patch(circle)
    ax.text(x, 8.3, str(i+1), ha='center', va='center', fontsize=11, 
            fontweight='bold', color='white', zorder=11)
    # Label
    ax.text(x, 7.9, label, ha='center', va='center', fontsize=11, 
            fontweight='bold', color=color)
    # Underline
    ax.plot([x-0.6, x+0.6], [7.7, 7.7], color=color, linewidth=3, solid_capstyle='round')

# ============================================================
# STAGE 1: DATA BOXES
# ============================================================
# Training Data
draw_box(ax, 0.3, 5.8, 2.4, 1.1, COLORS_PRO['bg_blue'], COLORS_PRO['primary'])
ax.text(1.5, 6.5, 'Training Data', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(1.5, 6.1, r'$\mathcal{D}_{train}$', ha='center', va='center', fontsize=10, color=COLORS_PRO['gray'])

# Calibration Data
draw_box(ax, 0.3, 4.3, 2.4, 1.1, COLORS_PRO['bg_green'], COLORS_PRO['secondary'])
ax.text(1.5, 5.0, 'Calibration Data', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(1.5, 4.6, r'$\mathcal{D}_{cal}$', ha='center', va='center', fontsize=10, color=COLORS_PRO['gray'])

# Test Data
draw_box(ax, 0.3, 2.8, 2.4, 1.1, COLORS_PRO['bg_orange'], COLORS_PRO['accent'])
ax.text(1.5, 3.5, 'Test Data', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(1.5, 3.1, r'$\mathcal{D}_{test}$', ha='center', va='center', fontsize=10, color=COLORS_PRO['gray'])

# ============================================================
# STAGE 2: NEURAL NETWORK
# ============================================================
draw_box(ax, 3.2, 4.0, 2.6, 3.0, COLORS_PRO['bg_purple'], COLORS_PRO['purple'])
ax.text(4.5, 6.5, 'Neural Network', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(4.5, 5.9, r'$\hat{f}(X)$', ha='center', va='center', fontsize=14, fontweight='bold', color=COLORS_PRO['purple'])
ax.text(4.5, 5.2, r'$\mathcal{L} = MSE + \lambda \cdot \mathcal{L}_{phys}$', ha='center', va='center', fontsize=9, color=COLORS_PRO['gray'])

# Physics-Informed badge (below the box)
ax.text(4.5, 3.6, '⚡ Physics-Informed', ha='center', va='center', fontsize=9,
        color=COLORS_PRO['accent'], fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=COLORS_PRO['bg_orange'], 
                  edgecolor=COLORS_PRO['accent'], linewidth=1.5))

# ============================================================
# STAGE 3: CONFORMAL CALIBRATION
# ============================================================
draw_box(ax, 6.5, 4.0, 2.6, 3.0, COLORS_PRO['bg_green'], COLORS_PRO['secondary'])
ax.text(7.8, 6.5, 'Conformal', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(7.8, 6.0, 'Calibration', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(7.8, 5.2, r'$\hat{q}_{1-\alpha} = Q_{1-\alpha}(|S_i|)$', ha='center', va='center', fontsize=10, color=COLORS_PRO['secondary'])

# Nonconformity score (below the box)
ax.text(7.8, 3.6, r'$S_i = |Y_i - \hat{f}(X_i)|$', ha='center', va='center', fontsize=9,
        color=COLORS_PRO['secondary'], style='italic')

# ============================================================
# STAGE 4: PHYSICS PROJECTION
# ============================================================
draw_box(ax, 9.9, 4.0, 2.6, 3.0, COLORS_PRO['bg_orange'], COLORS_PRO['accent'])
ax.text(11.2, 6.5, 'Physics', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(11.2, 6.0, 'Projection', ha='center', va='center', fontsize=11, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(11.2, 5.2, r'$\Pi_{\mathcal{K}}(\hat{C}) = \hat{C} \cap \mathcal{K}$', ha='center', va='center', fontsize=10, color=COLORS_PRO['accent'])

# Constraint set (below the box)
ax.text(11.2, 3.6, r'$\mathcal{K} = [0, R_{max}]$', ha='center', va='center', fontsize=9,
        color=COLORS_PRO['accent'], style='italic')

# ============================================================
# STAGE 5: OUTPUT
# ============================================================
draw_box(ax, 13.0, 4.0, 2.4, 3.0, COLORS_PRO['white'], COLORS_PRO['secondary'], linewidth=3)
ax.text(14.2, 6.5, 'PCCP', ha='center', va='center', fontsize=12, fontweight='bold', color=COLORS_PRO['secondary'])
ax.text(14.2, 6.0, 'Interval', ha='center', va='center', fontsize=12, fontweight='bold', color=COLORS_PRO['secondary'])
ax.text(14.2, 5.2, r'$\hat{C}_{phys}(X)$', ha='center', va='center', fontsize=12, fontweight='bold', color=COLORS_PRO['dark'])
ax.text(14.2, 4.5, r'$[\hat{L}, \hat{U}]$', ha='center', va='center', fontsize=10, color=COLORS_PRO['gray'])

# ============================================================
# ARROWS - Clear flow connections
# ============================================================

# Arrow 1: Training Data → Neural Network
draw_arrow_simple(ax, 2.7, 6.35, 3.2, 6.0, COLORS_PRO['primary'])

# Arrow 2: Calibration Data → Neural Network (curved up)
draw_arrow_curved(ax, 2.7, 4.85, 3.2, 5.0, COLORS_PRO['secondary'], rad=0.3)

# Arrow 3: Neural Network → Conformal Calibration
draw_arrow_simple(ax, 5.8, 5.5, 6.5, 5.5, COLORS_PRO['purple'])

# Arrow 4: Calibration Data → Conformal Calibration (long curved)
draw_arrow_curved(ax, 2.7, 4.3, 6.5, 4.1, COLORS_PRO['secondary'], rad=0.55)

# Arrow 5: Conformal Calibration → Physics Projection
draw_arrow_simple(ax, 9.1, 5.5, 9.9, 5.5, COLORS_PRO['secondary'])

# Arrow 6: Test Data → Physics Projection (long curved at bottom)
draw_arrow_curved(ax, 2.7, 3.0, 9.9, 4, COLORS_PRO['accent'], rad=0.2)

# Arrow 7: Physics Projection → Output
draw_arrow_simple(ax, 12.5, 5.5, 13.0, 5.5, COLORS_PRO['accent'])

# ============================================================
# KEY PROPERTIES BOX (Bottom Left)
# ============================================================
draw_box(ax, 0.3, 0.4, 6.0, 1.8, COLORS_PRO['bg_gray'], COLORS_PRO['secondary'])
ax.text(0.6, 1.95, '✓ Key Properties (Theorem 1)', fontsize=11, fontweight='bold', 
        color=COLORS_PRO['secondary'], va='top')

props = [
    ('• Coverage:', r'$P(Y \in \hat{C}_{phys}) \geq 1 - \alpha$'),
    ('• Feasibility:', r'$\hat{C}_{phys} \subseteq \mathcal{K}$ (always)'),
    ('• Efficiency:', r'$|\hat{C}_{phys}| \leq |\hat{C}|$ (tighter)')
]

for i, (label, formula) in enumerate(props):
    y = 1.55 - i * 0.4
    ax.text(0.6, y, label, fontsize=10, fontweight='bold', color=COLORS_PRO['dark'], va='center')
    ax.text(2.0, y, formula, fontsize=10, color=COLORS_PRO['secondary'], va='center')

# ============================================================
# OUTPUT FORMULA BOX (Bottom Right)
# ============================================================
draw_box(ax, 7.0, 0.4, 8.7, 1.8, COLORS_PRO['white'], COLORS_PRO['light_gray'])
ax.text(11.35, 1.95, 'Output Interval Formula', fontsize=11, fontweight='bold', 
        color=COLORS_PRO['dark'], ha='center', va='top')
ax.text(11.35, 1.2, r'$\hat{C}_{phys}(X) = [\max(0, \hat{y} - \hat{q}), \min(R_{max}, \hat{y} + \hat{q})]$', 
        fontsize=12, fontweight='bold', color=COLORS_PRO['dark'], ha='center', va='center')
ax.text(11.35, 0.65, r'where $\hat{y} = \hat{f}(X)$ and $\hat{q}$ = calibrated quantile at level $(1-\alpha)$',
        fontsize=9, color=COLORS_PRO['gray'], ha='center', va='center', style='italic')

# ============================================================
# SAVE
# ============================================================
plt.tight_layout()
plt.savefig('Fig01_Framework.png', dpi=600, facecolor='white', edgecolor='none', 
            bbox_inches='tight', pad_inches=0.15)
plt.savefig('Fig01_Framework.pdf', dpi=600, facecolor='white', edgecolor='none',
            bbox_inches='tight', pad_inches=0.15)
plt.show()
print("✓ Figure 1: Framework (Fixed)")

### Figure 1b: Theorem 1 Visual

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Circle, Wedge, Polygon, Arc
from matplotlib.collections import PatchCollection
import matplotlib.patheffects as path_effects

# ============================================================
# PREMIUM COLOR PALETTE
# ============================================================
C = {
    # Primary colors
    'blue': '#0066CC',
    'blue_light': '#E6F2FF',
    'purple': '#6B5B95',
    'purple_light': '#F0EDF5',
    'green': '#2ECC71',
    'green_light': '#E8F8F0',
    'orange': '#E67E22',
    'orange_light': '#FDF2E6',
    'teal': '#1ABC9C',
    'teal_light': '#E8FAF7',
    
    # Neutrals
    'dark': '#1A1A2E',
    'gray': '#6B7280',
    'light': '#E5E7EB',
    'white': '#FFFFFF',
    'bg': '#FAFBFC',
    
    # Accents
    'red': '#E74C3C',
    'gold': '#F1C40F'
}

# ============================================================
# CREATE FIGURE
# ============================================================
fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 10)
ax.axis('off')
fig.patch.set_facecolor(C['white'])

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def draw_premium_box(ax, x, y, w, h, bg_color, border_color, lw=2.5):
    """Draw elegant box with subtle shadow"""
    # Shadow
    shadow = FancyBboxPatch((x+0.06, y-0.06), w, h,
                             boxstyle="round,pad=0.02,rounding_size=0.12",
                             facecolor='#00000008', edgecolor='none')
    ax.add_patch(shadow)
    # Main box
    box = FancyBboxPatch((x, y), w, h,
                          boxstyle="round,pad=0.02,rounding_size=0.12",
                          facecolor=bg_color, edgecolor=border_color, linewidth=lw)
    ax.add_patch(box)

def draw_arrow(ax, start, end, color, lw=2.5, rad=0):
    """Draw smooth arrow"""
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                               mutation_scale=15, connectionstyle=f'arc3,rad={rad}'))

def draw_icon_data(ax, cx, cy, color, size=0.3):
    """Draw database/data icon"""
    # Stack of rectangles
    for i in range(3):
        rect = FancyBboxPatch((cx-size, cy-size*0.3+i*size*0.35), size*2, size*0.3,
                               boxstyle="round,pad=0.01,rounding_size=0.05",
                               facecolor=color, edgecolor=C['white'], linewidth=1,
                               alpha=0.7+i*0.1)
        ax.add_patch(rect)

def draw_icon_neural(ax, cx, cy, color, size=0.25):
    """Draw neural network icon"""
    layers = [3, 4, 3]
    all_pos = []
    for i, n in enumerate(layers):
        for j in range(n):
            x = cx + (i-1) * size * 2
            y = cy + (j - (n-1)/2) * size * 0.8
            all_pos.append((x, y, i))
            circle = Circle((x, y), size*0.2, facecolor=color, 
                           edgecolor=C['white'], linewidth=1, alpha=0.9)
            ax.add_patch(circle)
    # Connections
    for p1 in all_pos:
        for p2 in all_pos:
            if p2[2] == p1[2] + 1:
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 
                       color=color, alpha=0.3, linewidth=0.6)

def draw_icon_chart(ax, cx, cy, color, size=0.3):
    """Draw histogram/chart icon"""
    bars = [0.4, 0.7, 1.0, 0.8, 0.5, 0.3]
    w = size * 0.25
    for i, h in enumerate(bars):
        rect = plt.Rectangle((cx - size*0.8 + i*w*1.2, cy - size*0.5), 
                              w, h*size, facecolor=color, alpha=0.7+i*0.05)
        ax.add_patch(rect)

def draw_icon_filter(ax, cx, cy, color, size=0.3):
    """Draw funnel/filter icon for projection"""
    # Funnel shape
    points = [(cx-size, cy+size*0.5), (cx+size, cy+size*0.5),
              (cx+size*0.3, cy-size*0.3), (cx-size*0.3, cy-size*0.3)]
    polygon = Polygon(points, facecolor=color, edgecolor=C['white'], 
                     linewidth=1.5, alpha=0.8)
    ax.add_patch(polygon)
    # Output
    rect = plt.Rectangle((cx-size*0.15, cy-size*0.7), size*0.3, size*0.4,
                          facecolor=color, edgecolor=C['white'], linewidth=1)
    ax.add_patch(rect)

def draw_icon_check(ax, cx, cy, color, size=0.3):
    """Draw checkmark in circle"""
    circle = Circle((cx, cy), size, facecolor=color, edgecolor=C['white'], linewidth=2)
    ax.add_patch(circle)
    # Checkmark
    ax.plot([cx-size*0.4, cx-size*0.1, cx+size*0.45], 
            [cy, cy-size*0.35, cy+size*0.4],
            color=C['white'], linewidth=3, solid_capstyle='round', solid_joinstyle='round')

# ============================================================
# TITLE SECTION
# ============================================================
# Main title with elegant styling
title = ax.text(9, 9.4, 'Physics-Constrained Conformal Prediction', 
                ha='center', va='center', fontsize=24, fontweight='bold', 
                color=C['dark'], family='serif')

# Subtitle
ax.text(9, 8.85, 'A Framework for Physically Feasible Uncertainty Quantification',
        ha='center', va='center', fontsize=14, color=C['gray'], style='italic')

# Decorative line
ax.plot([4, 14], [8.55, 8.55], color=C['light'], linewidth=2)
ax.plot([7, 11], [8.55, 8.55], color=C['teal'], linewidth=4, solid_capstyle='round')

# ============================================================
# STAGE INDICATORS (Top row)
# ============================================================
stages = [
    (1.8, '1', 'DATA', C['blue'], 'Data\nPartitioning'),
    (5.4, '2', 'MODEL', C['purple'], 'Neural Network\nTraining'),
    (9.0, '3', 'CALIBRATE', C['green'], 'Conformal\nQuantile'),
    (12.6, '4', 'PROJECT', C['orange'], 'Physics\nConstraints'),
    (16.2, '5', 'OUTPUT', C['teal'], 'Feasible\nIntervals')
]

for x, num, label, color, desc in stages:
    # Outer glow
    glow = Circle((x, 8.0), 0.32, facecolor=color, alpha=0.15)
    ax.add_patch(glow)
    # Main circle
    circle = Circle((x, 8.0), 0.25, facecolor=color, edgecolor=C['white'], linewidth=2.5)
    ax.add_patch(circle)
    # Number
    ax.text(x, 8.0, num, ha='center', va='center', fontsize=16, 
            fontweight='bold', color=C['white'])
    # Label
    ax.text(x, 7.5, label, ha='center', va='center', fontsize=14, 
            fontweight='bold', color=color)
    # Description
    ax.text(x, 7.1, desc, ha='center', va='center', fontsize=10, 
            color=C['gray'], linespacing=1.1)

# ============================================================
# STAGE 1: DATA (Left column)
# ============================================================
data_x = 0.4

# Training Data Box
draw_premium_box(ax, data_x, 4.8, 2.8, 1.4, C['blue_light'], C['blue'])
draw_icon_data(ax, data_x + 0.55, 5.5, C['blue'], 0.25)
ax.text(data_x + 1.7, 5.7, 'Training Data', ha='center', va='center', 
        fontsize=13, fontweight='bold', color=C['dark'])
ax.text(data_x + 1.7, 5.25, r'$\mathcal{D}_{train}$', ha='center', va='center', 
        fontsize=13, color=C['blue'], style='italic')

# Calibration Data Box
draw_premium_box(ax, data_x, 3.1, 2.8, 1.4, C['green_light'], C['green'])
draw_icon_data(ax, data_x + 0.55, 3.8, C['green'], 0.25)
ax.text(data_x + 1.7, 4.0, 'Calibration Data', ha='center', va='center', 
        fontsize=13, fontweight='bold', color=C['dark'])
ax.text(data_x + 1.7, 3.55, r'$\mathcal{D}_{cal}$', ha='center', va='center', 
        fontsize=13, color=C['green'], style='italic')

# Test Data Box  
draw_premium_box(ax, data_x, 1.4, 2.8, 1.4, C['orange_light'], C['orange'])
draw_icon_data(ax, data_x + 0.55, 2.1, C['orange'], 0.25)
ax.text(data_x + 1.7, 2.3, 'Test Data', ha='center', va='center', 
        fontsize=13, fontweight='bold', color=C['dark'])
ax.text(data_x + 1.7, 1.85, r'$\mathcal{D}_{test}$', ha='center', va='center', 
        fontsize=13, color=C['orange'], style='italic')

# ============================================================
# STAGE 2: NEURAL NETWORK
# ============================================================
nn_x, nn_y = 3.9, 2.6
draw_premium_box(ax, nn_x, nn_y, 3.0, 3.6, C['purple_light'], C['purple'])

ax.text(nn_x + 1.5, 5.9, 'Base Predictor', ha='center', va='center', 
        fontsize=12, fontweight='bold', color=C['gray'])
ax.text(nn_x + 1.5, 5.5, '(Any Regression Model)', ha='center', va='center', 
        fontsize=14, fontweight='bold', color=C['dark'])

# Neural network icon
draw_icon_neural(ax, nn_x + 1.5, 4.6, C['purple'], 0.28)

# Function notation
ax.text(nn_x + 1.5, 3.7, r'$\hat{f}(X)$', ha='center', va='center', 
        fontsize=18, fontweight='bold', color=C['purple'])

# ============================================================
# STAGE 3: CONFORMAL CALIBRATION
# ============================================================
cal_x, cal_y = 7.5, 2.6
draw_premium_box(ax, cal_x, cal_y, 3.0, 3.6, C['green_light'], C['green'])

ax.text(cal_x + 1.5, 5.9, 'Conformal', ha='center', va='center', 
        fontsize=14, fontweight='bold', color=C['dark'])
ax.text(cal_x + 1.5, 5.5, 'Calibration', ha='center', va='center', 
        fontsize=14, fontweight='bold', color=C['dark'])

# Histogram icon
draw_icon_chart(ax, cal_x + 1.5, 4.5, C['green'], 0.35)

# Quantile line indicator
ax.axvline(cal_x + 1.85, ymin=0.42, ymax=0.5, color=C['red'], linewidth=2.5, linestyle='-')
ax.text(cal_x + 2.15, 4.65, r'$\hat{q}$', fontsize=9, color=C['red'], fontweight='bold')

# Quantile formula
ax.text(cal_x + 1.5, 3.7, r'$\hat{q}_{1-\alpha}$', ha='center', va='center', 
        fontsize=16, fontweight='bold', color=C['green'])

# Score formula (small badge)
score_box = FancyBboxPatch((cal_x + 0.15, cal_y + 0.15), 2.7, 0.55,
                            boxstyle="round,pad=0.02,rounding_size=0.08",
                            facecolor=C['white'], edgecolor=C['green'],
                            linewidth=1, linestyle='--', alpha=0.9)
ax.add_patch(score_box)
ax.text(cal_x + 1.5, cal_y + 0.42, r'$S_i = |Y_i - \hat{f}(X_i)|$', 
        ha='center', va='center', fontsize=10, color=C['green'])

# ============================================================
# STAGE 4: PHYSICS PROJECTION
# ============================================================
proj_x, proj_y = 11.1, 2.6
draw_premium_box(ax, proj_x, proj_y, 3.0, 3.6, C['orange_light'], C['orange'])

ax.text(proj_x + 1.5, 6.05, 'Core Contribution', ha='center', va='center',
        fontsize=11, fontweight='bold', color=C['orange'])

ax.text(proj_x + 1.5, 5.7, 'Physics Projection', ha='center', va='center', 
        fontsize=15, fontweight='bold', color=C['dark'])

# Visualization: interval clipping
# Original interval (gray, extends past boundary)
ax.plot([proj_x + 0.3, proj_x + 2.7], [4.7, 4.7], color=C['light'], 
        linewidth=12, solid_capstyle='round', alpha=0.8)
# Infeasible part (red)
ax.plot([proj_x + 0.3, proj_x + 1.0], [4.7, 4.7], color=C['red'], 
        linewidth=12, solid_capstyle='round', alpha=0.4)
# Feasible projected (orange)
ax.plot([proj_x + 1.0, proj_x + 2.7], [4.3, 4.3], color=C['orange'], 
        linewidth=12, solid_capstyle='round', alpha=0.8)
# Boundary line
ax.plot([proj_x + 1.0, proj_x + 1.0], [4.0, 4.95], color=C['dark'], 
        linewidth=1.5, linestyle='--')
ax.text(proj_x + 1.0, 3.85, '0', ha='center', va='center', fontsize=11, color=C['dark'])

# Labels
ax.text(proj_x + 0.65, 4.95, 'Original', ha='center', va='center', fontsize=9, color=C['gray'])
ax.text(proj_x + 1.85, 4.05, 'Projected', ha='center', va='center', fontsize=9, 
        color=C['orange'], fontweight='bold')

# Projection formula
ax.text(proj_x + 1.5, 3.7, r'$\Pi_{\mathcal{K}}(\hat{C})$', ha='center', va='center', 
        fontsize=18, fontweight='bold', color=C['orange'])
bbox=dict(boxstyle='round,pad=0.25',
          facecolor=C['white'],
          edgecolor=C['orange'],
          linewidth=1.2)

# Constraint set (small badge)
const_box = FancyBboxPatch((proj_x + 0.15, proj_y + 0.15), 2.7, 0.55,
                            boxstyle="round,pad=0.02,rounding_size=0.08",
                            facecolor=C['white'], edgecolor=C['orange'],
                            linewidth=1, linestyle='--', alpha=0.9)
ax.add_patch(const_box)
ax.text(proj_x + 1.5, proj_y + 0.42,
        r'Physics-feasible set $\mathcal{K}$',
        ha='center', va='center', fontsize=10,
        color=C['orange'], fontweight='bold')

# ============================================================
# STAGE 5: OUTPUT
# ============================================================
out_x, out_y = 14.7, 2.6
draw_premium_box(ax, out_x, out_y, 2.9, 3.6, C['teal_light'], C['teal'], lw=3)

ax.text(out_x + 1.45, 5.9, 'PCCP', ha='center', va='center', 
        fontsize=16, fontweight='bold', color=C['teal'])
ax.text(out_x + 1.45, 5.45, 'Output', ha='center', va='center', 
        fontsize=14, fontweight='bold', color=C['dark'])

# Interval visualization
ax.plot([out_x + 0.35, out_x + 2.55], [4.6, 4.6], color=C['teal'], 
        linewidth=14, solid_capstyle='round', alpha=0.3)
ax.scatter([out_x + 1.45], [4.6], color=C['teal'], s=100, zorder=5, marker='o')
ax.text(out_x + 0.35, 4.3, r'$\hat{L}$', ha='center', va='center', fontsize=11, color=C['teal'])
ax.text(out_x + 1.45, 4.3, r'$\hat{y}$', ha='center', va='center', fontsize=11, color=C['dark'])
ax.text(out_x + 2.55, 4.3, r'$\hat{U}$', ha='center', va='center', fontsize=11, color=C['teal'])

# Output formula
ax.text(out_x + 1.45, 4, r'$\hat{C}_{phys}(X)$', ha='center', va='center', 
        fontsize=15, fontweight='bold', color=C['dark'])

# Properties checkmarks
props = ['Valid coverage', 'Always feasible', 'Tighter bounds']
for i, prop in enumerate(props):
    ax.text(out_x + 1.45, 3.6 - i*0.35, prop, ha='center', va='center', 
            fontsize=10, color=C['teal'], fontweight='bold')

# ============================================================
# ARROWS - Elegant flow connections
# ============================================================
lw = 2.5

# Training → NN
draw_arrow(ax, (3.2, 5.5), (3.9, 5.2), C['blue'], lw)

# Calibration → NN  
draw_arrow(ax, (3.2, 3.8), (3.9, 4.0), C['green'], lw, rad=0.25)

# NN → Calibration
draw_arrow(ax, (6.9, 4.4), (7.5, 4.4), C['purple'], lw)

# Calibration Data → Calibration (curved below)
draw_arrow(ax, (3.2, 3), (7.5, 2.5), C['green'], lw, rad=0.4)

# Calibration → Projection
draw_arrow(ax, (10.5, 4.4), (11.1, 4.4), C['green'], lw)

# Test Data → Projection (long curved)
draw_arrow(ax, (3.2, 1.8), (11.1, 2.6), C['orange'], lw, rad=0.15)

# Projection → Output
draw_arrow(ax, (14.1, 4.4), (14.7, 4.4), C['orange'], lw)

# ============================================================
# BOTTOM: THEOREM BOX
# ============================================================
draw_premium_box(ax, 0.4, 0.15, 8.0, 1.15, C['bg'], C['teal'], lw=2)

ax.text(1.1, 1.05, 'Theorem 1: Coverage Preservation', fontsize=13, fontweight='bold', 
         va='center')

# Properties with color indicators
thm_props = [
    ( 'Coverage:', r'$P(Y \in \hat{C}_{phys}) \geq 1-\alpha$'),
    ( 'Feasibility:', r'$\hat{C}_{phys} \subseteq \mathcal{K}$'),
    ( 'Efficiency:', r'$|\hat{C}_{phys}| \leq |\hat{C}|$')
]

for i, ( label, formula) in enumerate(thm_props):
    x_base = 0.7 + i * 2.8
    ax.plot([x_base, x_base + 0.12], [0.55, 0.55], linewidth=5, solid_capstyle='round')
    ax.text(x_base + 0.22, 0.55, label, fontsize=11, fontweight='bold', va='center')
    ax.text(x_base + 1, 0.55, formula, fontsize=11, va='center')

# ============================================================
# BOTTOM: FORMULA BOX
# ============================================================
draw_premium_box(ax, 8.8, 0.15, 8.8, 1.15, C['white'], C['light'], lw=1.5)

ax.text(13.2, 1.05, 'Output Formula', fontsize=13, fontweight='bold', 
        color=C['dark'], ha='center', va='center')

ax.text(13.2, 0.55, r'$\hat{C}_{phys}(X) = \left[\max(0, \hat{y}-\hat{q}),\ \min(R_{max}, \hat{y}+\hat{q})\right]$',
        fontsize=13, color=C['dark'], ha='center', va='center', fontweight='bold')

# ============================================================
# SAVE
# ============================================================
plt.tight_layout()
plt.savefig('Fig01_Framework.png', dpi=600, facecolor='white', edgecolor='none', 
            bbox_inches='tight', pad_inches=0.15)
plt.savefig('Fig01_Framework.pdf', dpi=600, facecolor='white', edgecolor='none',
            bbox_inches='tight', pad_inches=0.15)
plt.show()
print("✓ Figure 1: Premium Professional Framework")

### Figure 2: Sensor Data Exploration

In [ ]:
# ============================================================
# FIGURE 2: DATA EXPLORATION - Sensor Readings
# ============================================================

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 3, hspace=0.35, wspace=0.3)

# Select one engine for visualization
engine_id = 1
engine_data = train_df_raw[train_df_raw['unit_id'] == engine_id].sort_values('time')

# Key sensors to plot
sensors = ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13']
sensor_names = ['Sensor 2\n(T24)', 'Sensor 3\n(T30)', 'Sensor 4\n(T50)', 
                'Sensor 7\n(Ps30)', 'Sensor 8\n(phi)', 'Sensor 9\n(NRf)',
                'Sensor 11\n(NRc)', 'Sensor 12\n(BPR)', 'Sensor 13\n(farB)']

for i, (sensor, name) in enumerate(zip(sensors, sensor_names)):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    
    time = engine_data['time'].values
    values = engine_data[sensor].values
    
    # Color gradient based on degradation
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(time)))
    
    for j in range(len(time) - 1):
        ax.plot(time[j:j+2], values[j:j+2], color=colors[j], linewidth=2)
    
    ax.scatter(time[0], values[0], color=COLORS['pccp'], s=80, zorder=5, 
               marker='o', edgecolor='white', linewidth=2, label='Start')
    ax.scatter(time[-1], values[-1], color=COLORS['infeasible'], s=80, zorder=5,
               marker='X', edgecolor='white', linewidth=2, label='Failure')
    
    ax.set_xlabel('Time (cycles)', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if i == 0:
        ax.legend(loc='upper left', fontsize=8)

plt.suptitle(f'Figure 2: Sensor Degradation Patterns (Engine {engine_id})', 
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig('Fig02_Sensor_Patterns.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig02_Sensor_Patterns.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 2: Sensor Degradation Patterns")

### Figure 3: RUL Distribution

In [ ]:
# ============================================================
# FIGURE 3: RUL DISTRIBUTION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel (a): Train RUL distribution
ax1 = axes[0]
ax1.hist(y_train, bins=50, color=COLORS['cp'], alpha=0.7, edgecolor='white', linewidth=1.2)
ax1.axvline(MAX_RUL, color=COLORS['infeasible'], linestyle='--', linewidth=2, label=f'RUL cap = {MAX_RUL}')
ax1.axvline(y_train.mean(), color=COLORS['warning'], linestyle='-', linewidth=2, 
            label=f'Mean = {y_train.mean():.1f}')
ax1.set_xlabel('RUL (cycles)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('(a) Training Set Distribution', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Panel (b): Test RUL distribution
ax2 = axes[1]
ax2.hist(y_test, bins=50, color=COLORS['pccp'], alpha=0.7, edgecolor='white', linewidth=1.2)
ax2.axvline(MAX_RUL, color=COLORS['infeasible'], linestyle='--', linewidth=2)
ax2.axvline(y_test.mean(), color=COLORS['warning'], linestyle='-', linewidth=2,
            label=f'Mean = {y_test.mean():.1f}')
ax2.set_xlabel('RUL (cycles)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('(b) Test Set Distribution', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Panel (c): Critical region highlight
ax3 = axes[2]
colors_hist = [COLORS['pccp'] if x >= 30 else COLORS['warning'] if x >= 15 else COLORS['infeasible'] 
               for x in np.linspace(0, MAX_RUL, 50)]
n, bins, patches = ax3.hist(y_test, bins=50, edgecolor='white', linewidth=1.2)
for patch, c in zip(patches, colors_hist):
    patch.set_facecolor(c)
    patch.set_alpha(0.8)

ax3.axvline(30, color='black', linestyle='--', linewidth=2)
ax3.axvline(15, color='black', linestyle=':', linewidth=2)
ax3.text(8, ax3.get_ylim()[1]*0.9, 'Critical\n(RUL<15)', fontsize=9, ha='center', fontweight='bold')
ax3.text(22, ax3.get_ylim()[1]*0.9, 'Warning\n(15-30)', fontsize=9, ha='center', fontweight='bold')
ax3.text(60, ax3.get_ylim()[1]*0.9, 'Normal\n(RUL>30)', fontsize=9, ha='center', fontweight='bold')

ax3.set_xlabel('RUL (cycles)', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.set_title('(c) Critical Region Highlight', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.suptitle('Figure 3: RUL Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig03_RUL_Distribution.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig03_RUL_Distribution.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 3: RUL Distribution Analysis")

### Figure 4: Training Curves

In [ ]:
# ============================================================
# FIGURE 4: TRAINING CURVES
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Panel (a): Loss curves
ax1 = axes[0]
ax1.plot(epochs, history['train_loss'], color=COLORS['cp'], linewidth=2, label='Training Loss')
ax1.plot(epochs, history['val_loss'], color=COLORS['pccp'], linewidth=2, label='Validation Loss')
best_epoch = np.argmin(history['val_rmse']) + 1
ax1.axvline(best_epoch, color=COLORS['infeasible'], linestyle='--', linewidth=1.5, 
            label=f'Best Epoch ({best_epoch})')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('(a) Training & Validation Loss', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Panel (b): RMSE curve
ax2 = axes[1]
ax2.plot(epochs, history['val_rmse'], color=COLORS['purple'], linewidth=2.5)
ax2.scatter([best_epoch], [min(history['val_rmse'])], color=COLORS['infeasible'], 
            s=150, zorder=5, marker='*', edgecolor='white', linewidth=2)
ax2.annotate(f'Best: {min(history["val_rmse"]):.2f}', 
             xy=(best_epoch, min(history['val_rmse'])),
             xytext=(best_epoch + 10, min(history['val_rmse']) + 2),
             fontsize=10, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLORS['dark']))
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('RMSE (cycles)', fontsize=12)
ax2.set_title('(b) Validation RMSE', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Panel (c): Learning rate
ax3 = axes[2]
ax3.plot(epochs, history['lr'], color=COLORS['orange'], linewidth=2.5)
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('Learning Rate', fontsize=12)
ax3.set_title('(c) Learning Rate Schedule', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_yscale('log')

plt.suptitle('Figure 4: Training Progress', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig04_Training_Curves.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig04_Training_Curves.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 4: Training Progress")

### Figure 5: CP vs PCCP Comparison (6 panels)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.lines import Line2D

# ============================================================
# PREPARE DATA
# ============================================================

# --- Single Engine Trajectory Data ---
engine = viz_units_data[0]
with torch.no_grad():
    model.eval()
    pred_viz = model(torch.FloatTensor(engine['X']).to(device)).cpu().numpy()

t, y_true = engine['t'], engine['y']
lower_cp = pred_viz - q_90
upper_cp = pred_viz + q_90
lower_pccp = np.maximum(lower_cp, 0)
upper_pccp = np.minimum(upper_cp, MAX_RUL)

# Define critical region for single engine
critical_mask = y_true < 30
if critical_mask.sum() < 10:
    critical_mask = np.zeros(len(y_true), dtype=bool)
    critical_mask[-int(len(y_true) * 0.3):] = True

t_crit = t[critical_mask]
y_crit = y_true[critical_mask]
pred_crit = pred_viz[critical_mask]
lower_cp_crit = lower_cp[critical_mask]
upper_cp_crit = upper_cp[critical_mask]
lower_pccp_crit = lower_pccp[critical_mask]
upper_pccp_crit = upper_pccp[critical_mask]

# --- Test Set Critical Region Data (for scatter plots) ---
critical_mask_test = y_test < 30
y_crit_test = y_test[critical_mask_test]
pred_crit_test = results_pccp_90['pred'][critical_mask_test]
lower_cp_crit_test = results_cp_90['lower'][critical_mask_test]
upper_cp_crit_test = results_cp_90['upper'][critical_mask_test]
lower_pccp_crit_test = results_pccp_90['lower'][critical_mask_test]
upper_pccp_crit_test = results_pccp_90['upper'][critical_mask_test]
lower_raw_crit_test = results_cp_90['lower_raw'][critical_mask_test]

# ============================================================
# CREATE FIGURE (6 PANELS) - FIXED LAYOUT
# ============================================================
fig = plt.figure(figsize=(14, 16))
gs = gridspec.GridSpec(3, 2, height_ratios=[1, 1, 1], hspace=0.30, wspace=0.22)

# Color scheme
CP_COLOR = COLORS['cp']        # Blue
PCCP_COLOR = COLORS['pccp']    # Green
INFEAS_COLOR = COLORS['infeasible']  # Red
NEUTRAL_COLOR = COLORS['neutral']    # Gray

# ============================================================
# ROW 1: FULL DEGRADATION TRAJECTORY
# ============================================================

# ===== Panel (a): Standard CP - Full Trajectory =====
ax1 = fig.add_subplot(gs[0, 0])

ax1.fill_between(t, lower_cp, upper_cp, alpha=0.25, color=CP_COLOR, label='90% Prediction Interval')
ax1.plot(t, pred_viz, color=CP_COLOR, linewidth=2, label='Point Prediction $\\hat{y}$')
ax1.plot(t, y_true, color=NEUTRAL_COLOR, linewidth=2.5, linestyle='--', label='True RUL')
ax1.axhline(y=0, color=INFEAS_COLOR, linewidth=2, linestyle=':', label='Physical Boundary ($y=0$)')

neg_mask_full = lower_cp < 0
if np.any(neg_mask_full):
    ax1.fill_between(t, lower_cp, 0, where=neg_mask_full, alpha=0.5, color=INFEAS_COLOR, 
                     hatch='///', label='Infeasible Region')

ax1.axvspan(t_crit.min(), t_crit.max(), alpha=0.08, color='orange')

# Annotation - pointing to infeasible region
ax1.annotate('Prediction interval\nextends below zero', 
             xy=(t[neg_mask_full][len(t[neg_mask_full])//2], lower_cp[neg_mask_full].mean()),
             xytext=(t[len(t)//2], -20),
             fontsize=9, ha='center',
             arrowprops=dict(arrowstyle='->', color=INFEAS_COLOR, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.9))

ax1.set_xlabel('Time (cycles)', fontsize=11)
ax1.set_ylabel('RUL (cycles)', fontsize=11)
ax1.set_title('(a) Standard CP - Full Degradation Trajectory', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8, framealpha=0.95)
ax1.set_ylim(min(lower_cp.min() - 15, -40), max(upper_cp.max() + 10, 150))
ax1.grid(True, alpha=0.3)

neg_pct_full = 100 * np.mean(lower_cp < 0)
stats_text = f'Infeasible: {neg_pct_full:.1f}%\nMin lower bound: {lower_cp.min():.1f}'
ax1.text(0.03, 0.03, stats_text, transform=ax1.transAxes, fontsize=10,
         color=INFEAS_COLOR, fontweight='bold', verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.95))

# ===== Panel (b): PCCP - Full Trajectory =====
ax2 = fig.add_subplot(gs[0, 1])

ax2.fill_between(t, lower_pccp, upper_pccp, alpha=0.25, color=PCCP_COLOR, label='90% Prediction Interval')
ax2.plot(t, np.clip(pred_viz, 0, MAX_RUL), color=PCCP_COLOR, linewidth=2, label='Point Prediction $\\hat{y}$')
ax2.plot(t, y_true, color=NEUTRAL_COLOR, linewidth=2.5, linestyle='--', label='True RUL')
ax2.axhline(y=0, color=NEUTRAL_COLOR, linewidth=1.5, alpha=0.5, label='Physical Boundary ($y=0$)')

ax2.axvspan(t_crit.min(), t_crit.max(), alpha=0.08, color='orange')

# MOVED DOWN - Annotation below the curves
ax2.annotate('All intervals\nconstrained to $[0, R_{max}]$', 
             xy=(t[-10], lower_pccp[-10]),
             xytext=(t[int(len(t)*0.65)], -20),  # Moved to y=-20
             fontsize=9, ha='center',
             arrowprops=dict(arrowstyle='->', color=PCCP_COLOR, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.9))

ax2.set_xlabel('Time (cycles)', fontsize=11)
ax2.set_ylabel('RUL (cycles)', fontsize=11)
ax2.set_title('(b) PCCP - Full Degradation Trajectory', fontsize=12, fontweight='bold')
ax2.legend(loc='upper right', fontsize=8, framealpha=0.95)
ax2.set_ylim(min(lower_cp.min() - 15, -40), max(upper_cp.max() + 10, 150))
ax2.grid(True, alpha=0.3)

stats_text2 = f'Infeasible: 0.0%\nMin lower bound: {lower_pccp.min():.1f}'
ax2.text(0.03, 0.03, stats_text2, transform=ax2.transAxes, fontsize=10,
         color=PCCP_COLOR, fontweight='bold', verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95))

# ============================================================
# ROW 2: CRITICAL REGION (ZOOMED)
# ============================================================

# ===== Panel (c): Standard CP - Critical Region =====
ax3 = fig.add_subplot(gs[1, 0])

ax3.fill_between(t_crit, lower_cp_crit, upper_cp_crit, alpha=0.25, color=CP_COLOR, 
                 label='90% Prediction Interval')
ax3.plot(t_crit, pred_crit, color=CP_COLOR, linewidth=2, label='Point Prediction')
ax3.plot(t_crit, y_crit, color=NEUTRAL_COLOR, linewidth=2.5, linestyle='--', label='True RUL')
ax3.axhline(y=0, color=INFEAS_COLOR, linewidth=2.5, linestyle='-', label='Physical Boundary')

neg_mask_crit = lower_cp_crit < 0
if np.any(neg_mask_crit):
    ax3.fill_between(t_crit, lower_cp_crit, 0, where=neg_mask_crit, 
                     alpha=0.5, color=INFEAS_COLOR, hatch='///')
    ax3.scatter(t_crit[neg_mask_crit], lower_cp_crit[neg_mask_crit], 
                color=INFEAS_COLOR, s=50, zorder=5, marker='x', linewidth=2,
                label='Infeasible Points')

if neg_mask_crit.any():
    worst_idx = np.argmin(lower_cp_crit)
    ax3.annotate(f'Worst violation:\n{lower_cp_crit[worst_idx]:.1f} cycles', 
                 xy=(t_crit[worst_idx], lower_cp_crit[worst_idx]),
                 xytext=(t_crit[worst_idx]-5, lower_cp_crit[worst_idx]-8),
                 fontsize=9, ha='center',
                 arrowprops=dict(arrowstyle='->', color=INFEAS_COLOR, lw=1.5),
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=INFEAS_COLOR))

ax3.set_xlabel('Time (cycles)', fontsize=11)
ax3.set_ylabel('RUL (cycles)', fontsize=11)
ax3.set_title('(c) Standard CP - Critical Region Detail (RUL < 30)', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=8, framealpha=0.95)
ax3.set_ylim(min(lower_cp_crit.min() - 12, -35), max(upper_cp_crit.max() + 5, 60))
ax3.grid(True, alpha=0.3)

neg_pct_crit = 100 * neg_mask_crit.mean()
stats_text3 = f'Infeasible: {neg_pct_crit:.1f}%\nMin lower: {lower_cp_crit.min():.1f}'
ax3.text(0.03, 0.03, stats_text3, transform=ax3.transAxes, fontsize=10,
         verticalalignment='bottom', fontweight='bold', color=INFEAS_COLOR,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.95))

# ===== Panel (d): PCCP - Critical Region =====
ax4 = fig.add_subplot(gs[1, 1])

ax4.fill_between(t_crit, lower_pccp_crit, upper_pccp_crit, alpha=0.25, color=PCCP_COLOR,
                 label='90% Prediction Interval')
ax4.plot(t_crit, np.clip(pred_crit, 0, MAX_RUL), color=PCCP_COLOR, linewidth=2, label='Point Prediction')
ax4.plot(t_crit, y_crit, color=NEUTRAL_COLOR, linewidth=2.5, linestyle='--', label='True RUL')
ax4.axhline(y=0, color=NEUTRAL_COLOR, linewidth=1.5, linestyle='-', alpha=0.5, label='Physical Boundary')

ax4.fill_between(t_crit, 0, lower_pccp_crit, alpha=0.15, color=PCCP_COLOR,
                 label='Feasible Lower Bound')

# MOVED DOWN - Annotation below the curve
ax4.annotate('Lower bound\nprojected to 0', 
             xy=(t_crit[-5], 0),
             xytext=(t_crit[len(t_crit)//2], -18),  # Moved to y=-18
             fontsize=9, ha='center',
             arrowprops=dict(arrowstyle='->', color=PCCP_COLOR, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=PCCP_COLOR))

ax4.set_xlabel('Time (cycles)', fontsize=11)
ax4.set_ylabel('RUL (cycles)', fontsize=11)
ax4.set_title('(d) PCCP - Critical Region Detail (RUL < 30)', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right', fontsize=8, framealpha=0.95)
ax4.set_ylim(min(lower_cp_crit.min() - 12, -35), max(upper_cp_crit.max() + 5, 60))
ax4.grid(True, alpha=0.3)

stats_text4 = f'Infeasible: 0.0%\nMin lower: {lower_pccp_crit.min():.1f}'
ax4.text(0.03, 0.03, stats_text4, transform=ax4.transAxes, fontsize=10,
         verticalalignment='bottom', fontweight='bold', color=PCCP_COLOR,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95))

# ============================================================
# ROW 3: SCATTER PLOTS - FIXED SIZE (NO aspect='equal')
# ============================================================

# ===== Panel (e): Standard CP - Scatter Plot =====
ax5 = fig.add_subplot(gs[2, 0])

infeasible_cp = lower_raw_crit_test < 0
covered_cp = (y_crit_test >= lower_cp_crit_test) & (y_crit_test <= upper_cp_crit_test)

# Plot error bars
for i in range(0, len(y_crit_test), 2):
    color = INFEAS_COLOR if infeasible_cp[i] else CP_COLOR
    alpha = 0.6 if infeasible_cp[i] else 0.25
    ax5.plot([y_crit_test[i], y_crit_test[i]], 
             [lower_cp_crit_test[i], upper_cp_crit_test[i]], 
             color=color, alpha=alpha, linewidth=0.8)

ax5.scatter(y_crit_test[~infeasible_cp], pred_crit_test[~infeasible_cp], 
            c=CP_COLOR, s=20, alpha=0.5, label='Feasible Intervals')
ax5.scatter(y_crit_test[infeasible_cp], pred_crit_test[infeasible_cp], 
            c=INFEAS_COLOR, s=30, alpha=0.8, marker='x', linewidth=1.5,
            label='Infeasible Intervals (lower < 0)')

ax5.plot([0, 30], [0, 30], 'k--', linewidth=2, label='Perfect Prediction')
ax5.axhline(0, color=INFEAS_COLOR, linestyle=':', linewidth=1.5, alpha=0.7)

ax5.set_xlabel('True RUL (cycles)', fontsize=11)
ax5.set_ylabel('Predicted RUL (cycles)', fontsize=11)
ax5.set_title(f'(e) Standard CP - All Test Samples (n={len(y_crit_test)})', fontsize=12, fontweight='bold')
ax5.legend(loc='upper left', fontsize=8, framealpha=0.95)
ax5.set_xlim(-2, 32)
ax5.set_ylim(-20, 50)
ax5.grid(True, alpha=0.3)
# REMOVED: ax5.set_aspect('equal', adjustable='box')

cp_coverage = 100 * covered_cp.mean()
cp_infeasible_pct = 100 * infeasible_cp.mean()

# Annotation in bottom area
ax5.text(5, -15, f'{cp_infeasible_pct:.0f}% of intervals\nhave lower < 0', 
         fontsize=9, ha='center', color=INFEAS_COLOR, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.9))

stats_text5 = f'Coverage: {cp_coverage:.1f}%\nInfeasible: {cp_infeasible_pct:.1f}%\nMin lower: {lower_raw_crit_test.min():.1f}'
ax5.text(0.97, 0.03, stats_text5, transform=ax5.transAxes, fontsize=9, ha='right', va='bottom',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=CP_COLOR, alpha=0.95))

# ===== Panel (f): PCCP - Scatter Plot =====
ax6 = fig.add_subplot(gs[2, 1])

covered_pccp = (y_crit_test >= lower_pccp_crit_test) & (y_crit_test <= upper_pccp_crit_test)

for i in range(0, len(y_crit_test), 2):
    ax6.plot([y_crit_test[i], y_crit_test[i]], 
             [lower_pccp_crit_test[i], upper_pccp_crit_test[i]], 
             color=PCCP_COLOR, alpha=0.25, linewidth=0.8)

ax6.scatter(y_crit_test, pred_crit_test, c=PCCP_COLOR, s=20, alpha=0.5, 
            label='All Intervals Feasible')

ax6.plot([0, 30], [0, 30], 'k--', linewidth=2, label='Perfect Prediction')
ax6.axhline(0, color=NEUTRAL_COLOR, linestyle='-', linewidth=1, alpha=0.5)

ax6.set_xlabel('True RUL (cycles)', fontsize=11)
ax6.set_ylabel('Predicted RUL (cycles)', fontsize=11)
ax6.set_title(f'(f) PCCP - All Test Samples (n={len(y_crit_test)})', fontsize=12, fontweight='bold')
ax6.legend(loc='upper left', fontsize=8, framealpha=0.95)
ax6.set_xlim(-2, 32)
ax6.set_ylim(-20, 50)
ax6.grid(True, alpha=0.3)
# REMOVED: ax6.set_aspect('equal', adjustable='box')

pccp_coverage = 100 * covered_pccp.mean()

# Annotation in bottom area
ax6.text(5, -15, '100% of intervals\nare physically valid', 
         fontsize=9, ha='center', color=PCCP_COLOR, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.9))

stats_text6 = f'Coverage: {pccp_coverage:.1f}%\nInfeasible: 0.0%\nMin lower: {lower_pccp_crit_test.min():.1f}'
ax6.text(0.97, 0.03, stats_text6, transform=ax6.transAxes, fontsize=9, ha='right', va='bottom',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95))

# ============================================================
# SAVE FIGURE
# ============================================================
plt.tight_layout()
plt.savefig('Fig02_CP_vs_PCCP_Complete.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig02_CP_vs_PCCP_Complete.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()

# ============================================================
# PRINT SUMMARY
# ============================================================
print("="*70)
print("✓ Figure 2 saved (6 panels)")
print("="*70)
print(f"\n📊 PANEL SUMMARY:")
print(f"   (a) CP Full Trajectory:      {len(t)} points, Infeasible: {neg_pct_full:.1f}%")
print(f"   (b) PCCP Full Trajectory:    {len(t)} points, Infeasible: 0.0%")
print(f"   (c) CP Critical Region:      {len(t_crit)} points, Infeasible: {neg_pct_crit:.1f}%")
print(f"   (d) PCCP Critical Region:    {len(t_crit)} points, Infeasible: 0.0%")
print(f"   (e) CP Test Scatter:         {len(y_crit_test)} points, Infeasible: {cp_infeasible_pct:.1f}%")
print(f"   (f) PCCP Test Scatter:       {len(y_crit_test)} points, Infeasible: 0.0%")
print(f"\n🎯 KEY METRICS:")
print(f"   Coverage preserved: CP={cp_coverage:.1f}% ≈ PCCP={pccp_coverage:.1f}%")
print(f"   Physical violations eliminated: {cp_infeasible_pct:.1f}% → 0.0%")
print("="*70)

### Figure 6: Multiple Engines Comparison

In [ ]:
# ============================================================
# FIGURE 6: MULTIPLE ENGINES COMPARISON
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, eng in enumerate(viz_units_data[:6]):
    ax = axes[idx]
    
    with torch.no_grad():
        pred_e = model(torch.FloatTensor(eng['X']).to(device)).cpu().numpy()
    
    t_e, y_e = eng['t'], eng['y']
    lower_e = np.maximum(pred_e - q_90, 0)
    upper_e = np.minimum(pred_e + q_90, MAX_RUL)
    
    ax.fill_between(t_e, lower_e, upper_e, alpha=0.3, color=COLORS['pccp'])
    ax.plot(t_e, np.clip(pred_e, 0, MAX_RUL), color=COLORS['pccp'], linewidth=2, label='PCCP')
    ax.plot(t_e, y_e, color=COLORS['neutral'], linewidth=2, linestyle='--', label='True')
    ax.axhline(y=0, color=COLORS['neutral'], linewidth=1, alpha=0.5)
    
    # Coverage for this engine
    covered = (y_e >= lower_e) & (y_e <= upper_e)
    cov_pct = 100 * covered.mean()
    
    ax.set_title(f'Engine {eng["unit_id"]} (Cov: {cov_pct:.0f}%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time', fontsize=10)
    ax.set_ylabel('RUL', fontsize=10)
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9)

plt.suptitle('Figure 6: PCCP Predictions Across Multiple Engines', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig06_Multiple_Engines.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig06_Multiple_Engines.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 6: Multiple Engines")

### Figure 7: Two-Panel Comprehensive Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

# ============================================================
# COMPUTE METRICS ACROSS COVERAGE LEVELS
# ============================================================

alpha_values = [0.20, 0.15, 0.10, 0.05, 0.02]  # 80%, 85%, 90%, 95%, 98%
nominal_coverages = [100 * (1 - a) for a in alpha_values]

results_by_coverage = []

for alpha in alpha_values:
    # Calibrate
    q_alpha, _ = calibrate(model, X_cal, y_cal, alpha)
    
    # Predict with CP and PCCP
    res_cp = predict_intervals(model, X_test, q_alpha, apply_physics=False)
    res_pccp = predict_intervals(model, X_test, q_alpha, apply_physics=True)
    
    # Compute metrics
    # CP metrics
    covered_cp = (y_test >= res_cp['lower']) & (y_test <= res_cp['upper'])
    cp_picp = 100 * np.mean(covered_cp)
    cp_mpiw = np.mean(res_cp['upper'] - res_cp['lower'])
    cp_neg = 100 * np.mean(res_cp['lower_raw'] < 0)
    cp_phys = 100 - cp_neg
    
    # PCCP metrics
    covered_pccp = (y_test >= res_pccp['lower']) & (y_test <= res_pccp['upper'])
    pccp_picp = 100 * np.mean(covered_pccp)
    pccp_mpiw = np.mean(res_pccp['upper'] - res_pccp['lower'])
    pccp_neg = 0.0  # Always 0 by construction
    pccp_phys = 100.0  # Always 100 by construction
    
    # MPIW improvement
    mpiw_improvement = 100 * (cp_mpiw - pccp_mpiw) / cp_mpiw
    
    results_by_coverage.append({
        'nominal': 100 * (1 - alpha),
        'alpha': alpha,
        'q': q_alpha,
        'CP_PICP': cp_picp,
        'PCCP_PICP': pccp_picp,
        'CP_MPIW': cp_mpiw,
        'PCCP_MPIW': pccp_mpiw,
        'CP_Neg': cp_neg,
        'PCCP_Neg': pccp_neg,
        'CP_Phys': cp_phys,
        'PCCP_Phys': pccp_phys,
        'MPIW_Improvement': mpiw_improvement
    })

# Extract arrays for plotting
nominals = [r['nominal'] for r in results_by_coverage]
cp_picps = [r['CP_PICP'] for r in results_by_coverage]
pccp_picps = [r['PCCP_PICP'] for r in results_by_coverage]
cp_mpiws = [r['CP_MPIW'] for r in results_by_coverage]
pccp_mpiws = [r['PCCP_MPIW'] for r in results_by_coverage]
cp_negs = [r['CP_Neg'] for r in results_by_coverage]
pccp_negs = [r['PCCP_Neg'] for r in results_by_coverage]
cp_phys = [r['CP_Phys'] for r in results_by_coverage]
pccp_phys = [r['PCCP_Phys'] for r in results_by_coverage]
mpiw_improvements = [r['MPIW_Improvement'] for r in results_by_coverage]

print("📊 Results computed for coverage levels:", nominals)

# ============================================================
# FIGURE 3: TWO-PANEL COMPREHENSIVE PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color scheme
CP_COLOR = COLORS['cp']
PCCP_COLOR = COLORS['pccp']
INFEAS_COLOR = COLORS['infeasible']

# ===== Panel (a): Coverage Calibration & Interval Width =====
ax1 = axes[0]
ax1_twin = ax1.twinx()

# Coverage calibration (left y-axis)
ax1.plot(nominals, nominals, 'k--', linewidth=2, label='Ideal Coverage', zorder=1)
ax1.fill_between(nominals, np.array(nominals) - 2, np.array(nominals) + 2, 
                  alpha=0.1, color='gray', label='±2% Tolerance')
ax1.plot(nominals, cp_picps, 'o-', color=CP_COLOR, linewidth=2.5, markersize=10, 
         markeredgecolor='white', markeredgewidth=2, label='CP Coverage', zorder=3)
ax1.plot(nominals, pccp_picps, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='PCCP Coverage', zorder=3)

# Add value labels for coverage
for i, (nom, cp_p, pccp_p) in enumerate(zip(nominals, cp_picps, pccp_picps)):
    if i == 2:  # Label at 90%
        ax1.annotate(f'{cp_p:.1f}%', (nom, cp_p), textcoords="offset points", 
                     xytext=(0, 12), ha='center', fontsize=9, color=CP_COLOR, fontweight='bold')
        ax1.annotate(f'{pccp_p:.1f}%', (nom, pccp_p), textcoords="offset points", 
                     xytext=(0, -18), ha='center', fontsize=9, color=PCCP_COLOR, fontweight='bold')

# MPIW (right y-axis) - as bars in background
width = 2.5
ax1_twin.bar(np.array(nominals) - width/2, cp_mpiws, width, alpha=0.25, color=CP_COLOR, 
             label='CP Width', edgecolor=CP_COLOR, linewidth=1.5)
ax1_twin.bar(np.array(nominals) + width/2, pccp_mpiws, width, alpha=0.25, color=PCCP_COLOR,
             label='PCCP Width', edgecolor=PCCP_COLOR, linewidth=1.5)

# Annotations for MPIW improvement
for i, (nom, cp_m, pccp_m, impr) in enumerate(zip(nominals, cp_mpiws, pccp_mpiws, mpiw_improvements)):
    mid_y = (cp_m + pccp_m) / 2
    ax1_twin.annotate(f'-{impr:.0f}%', (nom, mid_y), ha='center', fontsize=8, 
                      color=PCCP_COLOR, fontweight='bold',
                      bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'))

ax1.set_xlabel('Nominal Coverage Level (%)', fontsize=12)
ax1.set_ylabel('Empirical Coverage (%)', fontsize=12, color='black')
ax1_twin.set_ylabel('Mean Prediction Interval Width (cycles)', fontsize=12, color='gray')
ax1.set_title('(a) Coverage Calibration & Interval Efficiency', fontsize=13, fontweight='bold')

ax1.set_xlim(78, 100)
ax1.set_ylim(75, 102)
ax1_twin.set_ylim(0, max(cp_mpiws) * 1.3)

ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='y', labelcolor='black')
ax1_twin.tick_params(axis='y', labelcolor='gray')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9, framealpha=0.95)

# Key finding annotation
ax1.text(0.03, 0.97, 'Theorem 1 Validated:\nCP ≈ PCCP coverage\nat all levels', 
         transform=ax1.transAxes, fontsize=10, va='top', ha='left',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95),
         color=PCCP_COLOR, fontweight='bold')

# ===== Panel (b): Physical Consistency =====
ax2 = axes[1]

# Physical violations (Negative %)
ax2.fill_between(nominals, 0, cp_negs, alpha=0.3, color=INFEAS_COLOR, label='CP Violations')
ax2.plot(nominals, cp_negs, 'o-', color=INFEAS_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='CP Negative %')
ax2.plot(nominals, pccp_negs, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='PCCP Negative %')

# Add value labels
for nom, cp_n in zip(nominals, cp_negs):
    ax2.annotate(f'{cp_n:.1f}%', (nom, cp_n), textcoords="offset points", 
                 xytext=(0, 10), ha='center', fontsize=10, color=INFEAS_COLOR, fontweight='bold')

# PCCP always 0% annotation
ax2.annotate('PCCP: 0% at all levels\n(by construction)', 
             xy=(90, 0), xytext=(85, 8),
             fontsize=10, ha='center', color=PCCP_COLOR, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=PCCP_COLOR, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95))

# Highlight trend
ax2.annotate('Higher coverage\n→ More violations', 
             xy=(96, cp_negs[-1]), xytext=(92, cp_negs[-1] + 5),
             fontsize=9, ha='center', color=INFEAS_COLOR,
             arrowprops=dict(arrowstyle='->', color=INFEAS_COLOR, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.9))

ax2.set_xlabel('Nominal Coverage Level (%)', fontsize=12)
ax2.set_ylabel('Physically Infeasible Intervals (%)', fontsize=12)
ax2.set_title('(b) Physical Consistency Across Coverage Levels', fontsize=13, fontweight='bold')

ax2.set_xlim(78, 100)
ax2.set_ylim(-2, max(cp_negs) * 1.4)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left', fontsize=10, framealpha=0.95)

# Key finding box
key_text = f'At 90% coverage:\n• CP: {cp_negs[2]:.1f}% infeasible\n• PCCP: 0% infeasible\n• Same coverage ({cp_picps[2]:.1f}%)'
ax2.text(0.97, 0.97, key_text, transform=ax2.transAxes, fontsize=10, va='top', ha='right',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', alpha=0.95))

plt.tight_layout()
plt.savefig('Fig03_Main_Results_Curves.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig03_Main_Results_Curves.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 3: Main Results Curves (2 panels)")

# ============================================================
# ALTERNATIVE: SINGLE COMPREHENSIVE FIGURE (4 subplots)
# ============================================================

fig2 = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.3, wspace=0.25)

# ===== Panel (a): Coverage Calibration =====
ax1 = fig2.add_subplot(gs[0, 0])

ax1.fill_between(nominals, np.array(nominals) - 2, np.array(nominals) + 2, 
                  alpha=0.15, color='gray', label='±2% Tolerance Band')
ax1.plot(nominals, nominals, 'k--', linewidth=2, label='Ideal (y=x)')
ax1.plot(nominals, cp_picps, 'o-', color=CP_COLOR, linewidth=2.5, markersize=12, 
         markeredgecolor='white', markeredgewidth=2, label='Standard CP')
ax1.plot(nominals, pccp_picps, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=12,
         markeredgecolor='white', markeredgewidth=2, label='PCCP')

# Value labels
for nom, cp_p, pccp_p in zip(nominals, cp_picps, pccp_picps):
    ax1.annotate(f'{cp_p:.1f}', (nom, cp_p), textcoords="offset points", 
                 xytext=(-15, 5), ha='center', fontsize=9, color=CP_COLOR)
    ax1.annotate(f'{pccp_p:.1f}', (nom, pccp_p), textcoords="offset points", 
                 xytext=(15, -5), ha='center', fontsize=9, color=PCCP_COLOR)

ax1.set_xlabel('Nominal Coverage (%)', fontsize=11)
ax1.set_ylabel('Empirical Coverage (%)', fontsize=11)
ax1.set_title('(a) Coverage Calibration', fontsize=12, fontweight='bold')
ax1.set_xlim(78, 100)
ax1.set_ylim(78, 100)
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Theorem 1 validation text
ax1.text(0.05, 0.95, 'Theorem 1: CP ≈ PCCP\n(coverage preserved)', 
         transform=ax1.transAxes, fontsize=10, va='top',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))

# ===== Panel (b): Interval Width (MPIW) =====
ax2 = fig2.add_subplot(gs[0, 1])

ax2.plot(nominals, cp_mpiws, 'o-', color=CP_COLOR, linewidth=2.5, markersize=12,
         markeredgecolor='white', markeredgewidth=2, label='Standard CP')
ax2.plot(nominals, pccp_mpiws, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=12,
         markeredgecolor='white', markeredgewidth=2, label='PCCP')
ax2.fill_between(nominals, pccp_mpiws, cp_mpiws, alpha=0.2, color=PCCP_COLOR,
                 label='Width Reduction')

# Improvement annotations
for nom, cp_m, pccp_m, impr in zip(nominals, cp_mpiws, pccp_mpiws, mpiw_improvements):
    mid = (cp_m + pccp_m) / 2
    ax2.annotate(f'-{impr:.0f}%', (nom, mid), ha='center', fontsize=9, 
                 color=PCCP_COLOR, fontweight='bold')

ax2.set_xlabel('Nominal Coverage (%)', fontsize=11)
ax2.set_ylabel('Mean Prediction Interval Width (cycles)', fontsize=11)
ax2.set_title('(b) Interval Efficiency', fontsize=12, fontweight='bold')
ax2.set_xlim(78, 100)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

# Corollary 1 text
ax2.text(0.95, 0.05, 'Corollary 1: PCCP ≤ CP\n(width reduction)', 
         transform=ax2.transAxes, fontsize=10, va='bottom', ha='right',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))

# ===== Panel (c): Physical Violations =====
ax3 = fig2.add_subplot(gs[1, 0])

ax3.fill_between(nominals, 0, cp_negs, alpha=0.3, color=INFEAS_COLOR)
ax3.plot(nominals, cp_negs, 'o-', color=INFEAS_COLOR, linewidth=2.5, markersize=12,
         markeredgecolor='white', markeredgewidth=2, label='Standard CP')
ax3.plot(nominals, pccp_negs, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=12,
         markeredgecolor='white', markeredgewidth=2, label='PCCP (always 0%)')

# Value labels for CP
for nom, neg in zip(nominals, cp_negs):
    ax3.annotate(f'{neg:.1f}%', (nom, neg), textcoords="offset points", 
                 xytext=(0, 10), ha='center', fontsize=10, color=INFEAS_COLOR, fontweight='bold')

ax3.set_xlabel('Nominal Coverage (%)', fontsize=11)
ax3.set_ylabel('Infeasible Intervals (%)', fontsize=11)
ax3.set_title('(c) Physical Violations (Negative Lower Bounds)', fontsize=12, fontweight='bold')
ax3.set_xlim(78, 100)
ax3.set_ylim(-1, max(cp_negs) * 1.3)
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

# Key insight
ax3.annotate('PCCP eliminates\nALL violations', xy=(90, 0), xytext=(85, max(cp_negs)*0.5),
             fontsize=10, ha='center', color=PCCP_COLOR, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=PCCP_COLOR, lw=2),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=PCCP_COLOR))

# ===== Panel (d): Summary Metrics at 90% =====
ax4 = fig2.add_subplot(gs[1, 1])

# Create comparison table as bar chart
metrics_names = ['Coverage\n(%)', 'MPIW\n(cycles)', 'Physical\nConsistency (%)', 'Infeasible\n(%)']
cp_values_norm = [cp_picps[2]/100, cp_mpiws[2]/60, cp_phys[2]/100, cp_negs[2]/25]  # Normalized
pccp_values_norm = [pccp_picps[2]/100, pccp_mpiws[2]/60, pccp_phys[2]/100, 0]

x_pos = np.arange(len(metrics_names))
width = 0.35

bars1 = ax4.bar(x_pos - width/2, cp_values_norm, width, label='Standard CP', 
                color=CP_COLOR, alpha=0.8, edgecolor='white')
bars2 = ax4.bar(x_pos + width/2, pccp_values_norm, width, label='PCCP', 
                color=PCCP_COLOR, alpha=0.8, edgecolor='white')

# Actual value labels
cp_actual = [f'{cp_picps[2]:.1f}%', f'{cp_mpiws[2]:.1f}', f'{cp_phys[2]:.1f}%', f'{cp_negs[2]:.1f}%']
pccp_actual = [f'{pccp_picps[2]:.1f}%', f'{pccp_mpiws[2]:.1f}', '100%', '0%']

for bar, val in zip(bars1, cp_actual):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, val,
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=CP_COLOR)
for bar, val in zip(bars2, pccp_actual):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, val,
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=PCCP_COLOR)

ax4.set_xticks(x_pos)
ax4.set_xticklabels(metrics_names, fontsize=10)
ax4.set_ylabel('Normalized Value', fontsize=11)
ax4.set_title('(d) Summary Comparison at 90% Coverage', fontsize=12, fontweight='bold')
ax4.set_ylim(0, 1.3)
ax4.legend(loc='upper right', fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')


plt.tight_layout()
plt.savefig('Fig03_Main_Results_4Panels.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig03_Main_Results_4Panels.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 3: Main Results (4 panels)")

# ============================================================
# PRINT SUMMARY TABLE
# ============================================================
print("\n" + "="*80)
print("📊 RESULTS SUMMARY TABLE")
print("="*80)
print(f"{'Coverage':<12} {'CP PICP':<12} {'PCCP PICP':<12} {'CP MPIW':<12} {'PCCP MPIW':<12} {'Improvement':<12} {'CP Neg%':<10}")
print("-"*80)
for r in results_by_coverage:
    print(f"{r['nominal']:.0f}%{'':<9} {r['CP_PICP']:.1f}%{'':<7} {r['PCCP_PICP']:.1f}%{'':<7} "
          f"{r['CP_MPIW']:.2f}{'':<7} {r['PCCP_MPIW']:.2f}{'':<7} {r['MPIW_Improvement']:.1f}%{'':<7} {r['CP_Neg']:.1f}%")
print("="*80)

### Figure 8: Two-Panel Version

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

# ============================================================
# COMPUTE METRICS ACROSS RUL THRESHOLDS
# ============================================================

# More granular thresholds for smooth curves
rul_thresholds = [125, 100, 75, 50, 40, 30, 25, 20, 15, 10, 5]

critical_results_detailed = []

for thresh in rul_thresholds:
    mask = y_test < thresh
    n_samples = mask.sum()
    
    if n_samples > 10:  # Need minimum samples
        # CP metrics
        cp_covered = (y_test[mask] >= results_cp_90['lower'][mask]) & \
                     (y_test[mask] <= results_cp_90['upper'][mask])
        cp_picp = 100 * cp_covered.mean()
        cp_mpiw = np.mean(results_cp_90['upper'][mask] - results_cp_90['lower'][mask])
        cp_neg = 100 * np.mean(results_cp_90['lower_raw'][mask] < 0)
        
        # PCCP metrics
        pccp_covered = (y_test[mask] >= results_pccp_90['lower'][mask]) & \
                       (y_test[mask] <= results_pccp_90['upper'][mask])
        pccp_picp = 100 * pccp_covered.mean()
        pccp_mpiw = np.mean(results_pccp_90['upper'][mask] - results_pccp_90['lower'][mask])
        
        # Width improvement
        mpiw_improvement = 100 * (cp_mpiw - pccp_mpiw) / cp_mpiw if cp_mpiw > 0 else 0
        
        critical_results_detailed.append({
            'threshold': thresh,
            'n': n_samples,
            'CP_PICP': cp_picp,
            'PCCP_PICP': pccp_picp,
            'CP_MPIW': cp_mpiw,
            'PCCP_MPIW': pccp_mpiw,
            'CP_Neg': cp_neg,
            'MPIW_Improvement': mpiw_improvement
        })

# Extract arrays
thresholds = [r['threshold'] for r in critical_results_detailed]
n_samples = [r['n'] for r in critical_results_detailed]
cp_picps = [r['CP_PICP'] for r in critical_results_detailed]
pccp_picps = [r['PCCP_PICP'] for r in critical_results_detailed]
cp_mpiws = [r['CP_MPIW'] for r in critical_results_detailed]
pccp_mpiws = [r['PCCP_MPIW'] for r in critical_results_detailed]
cp_negs = [r['CP_Neg'] for r in critical_results_detailed]
mpiw_improvements = [r['MPIW_Improvement'] for r in critical_results_detailed]

print(f"📊 Computed metrics for {len(thresholds)} RUL thresholds")
print(f"   Thresholds: {thresholds}")

# ============================================================
# COLOR SCHEME
# ============================================================
CP_COLOR = COLORS['cp']
PCCP_COLOR = COLORS['pccp']
INFEAS_COLOR = COLORS['infeasible']
WARNING_COLOR = COLORS['warning']
NEUTRAL_COLOR = COLORS['neutral']

# ============================================================
# FIGURE 4: TWO-PANEL VERSION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ===== Panel (a): Coverage & MPIW vs RUL Threshold =====
ax1 = axes[0]
ax1_twin = ax1.twinx()

# Coverage curves (left y-axis)
ax1.axhline(90, color='black', linestyle='--', linewidth=1.5, alpha=0.5, label='90% Target')
ax1.plot(thresholds, cp_picps, 'o-', color=CP_COLOR, linewidth=2.5, markersize=8,
         markeredgecolor='white', markeredgewidth=1.5, label='CP Coverage')
ax1.plot(thresholds, pccp_picps, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=8,
         markeredgecolor='white', markeredgewidth=1.5, label='PCCP Coverage')

# MPIW curves (right y-axis)
ax1_twin.plot(thresholds, cp_mpiws, 'o--', color=CP_COLOR, linewidth=2, markersize=6,
              alpha=0.5, label='CP MPIW')
ax1_twin.plot(thresholds, pccp_mpiws, 's--', color=PCCP_COLOR, linewidth=2, markersize=6,
              alpha=0.5, label='PCCP MPIW')
ax1_twin.fill_between(thresholds, pccp_mpiws, cp_mpiws, alpha=0.15, color=PCCP_COLOR)

# Annotations
# Mark key thresholds
for thresh_val in [50, 30, 15]:
    if thresh_val in thresholds:
        idx = thresholds.index(thresh_val)
        ax1.axvline(thresh_val, color='gray', linestyle=':', alpha=0.5)
        ax1.annotate(f'RUL<{thresh_val}', xy=(thresh_val, ax1.get_ylim()[1]*0.923),
                     fontsize=8, ha='center', color='gray')

# Key insight annotation
ax1.text(0.03, 0.03, 'Coverage preserved\nacross all thresholds\n(Theorem 1)', 
         transform=ax1.transAxes, fontsize=10, va='bottom', ha='left',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95),
         color=PCCP_COLOR, fontweight='bold')

ax1.set_xlabel('RUL Threshold (cycles)', fontsize=12)
ax1.set_ylabel('Empirical Coverage (%)', fontsize=12, color='black')
ax1_twin.set_ylabel('Mean Prediction Interval Width (cycles)', fontsize=12, color='gray')
ax1.set_title('(a) Coverage & Interval Width vs RUL Threshold', fontsize=13, fontweight='bold')

ax1.set_xlim(max(thresholds) + 5, min(thresholds) - 2)  # Reverse x-axis (approaching failure)
ax1.set_ylim(85, 105)
ax1_twin.set_ylim(0, max(cp_mpiws) * 1.3)

ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='y', labelcolor='black')
ax1_twin.tick_params(axis='y', labelcolor='gray')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8, framealpha=0.95)

# Add arrow showing direction
ax1.annotate('', xy=(20, 92), xytext=(80, 92),
             arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax1.text(50, 92.3, 'Approaching Failure', ha='center', fontsize=9, style='italic')

# ===== Panel (b): Physical Violations vs RUL Threshold =====
ax2 = axes[1]

# Main violation curve
ax2.fill_between(thresholds, 0, cp_negs, alpha=0.3, color=INFEAS_COLOR)
ax2.plot(thresholds, cp_negs, 'o-', color=INFEAS_COLOR, linewidth=3, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='CP Negative %')

# PCCP line at 0
ax2.plot(thresholds, [0]*len(thresholds), 's-', color=PCCP_COLOR, linewidth=3, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='PCCP Negative % (always 0)')

# Add value labels for CP
for i, (thresh, neg) in enumerate(zip(thresholds, cp_negs)):
    if thresh in [125, 50, 30, 15, 5]:  # Label key points
        offset = 8 if neg < 80 else -12
        ax2.annotate(f'{neg:.0f}%', (thresh, neg), textcoords="offset points", 
                     xytext=(0, offset), ha='center', fontsize=10, 
                     color=INFEAS_COLOR, fontweight='bold')

# Mark danger zones
ax2.axhspan(0, 25, alpha=0.1, color='green', label='Low Risk (<25%)')
ax2.axhspan(25, 50, alpha=0.1, color='yellow')
ax2.axhspan(50, 100, alpha=0.1, color='red', label='High Risk (>50%)')

# Vertical threshold markers
for thresh_val, label in [(50, 'Moderate'), (30, 'Critical'), (15, 'Severe')]:
    if thresh_val in thresholds:
        ax2.axvline(thresh_val, color='gray', linestyle=':', alpha=0.7)

# Key insight annotations
ax2.annotate('CP violations\nexplode near failure', 
             xy=(15, cp_negs[thresholds.index(15)] if 15 in thresholds else 80),
             xytext=(50, 90),
             fontsize=10, ha='center', color=INFEAS_COLOR, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=INFEAS_COLOR, lw=2),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.95))

ax2.annotate('PCCP: 0% at ALL thresholds\n(by construction)', 
             xy=(60, 0), xytext=(80, 25),
             fontsize=10, ha='center', color=PCCP_COLOR, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=PCCP_COLOR, lw=2),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PCCP_COLOR, alpha=0.95))

ax2.set_xlabel('RUL Threshold (cycles)', fontsize=12)
ax2.set_ylabel('Physically Infeasible Intervals (%)', fontsize=12)
ax2.set_title('(b) Physical Violations vs RUL Threshold', fontsize=13, fontweight='bold')

ax2.set_xlim(max(thresholds) + 5, min(thresholds) - 2)  # Reverse x-axis
ax2.set_ylim(-5, 110)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left', fontsize=9, framealpha=0.95)

# Add sample size info
ax2_twin2 = ax2.twiny()
ax2_twin2.set_xlim(ax2.get_xlim())
ax2_twin2.set_xticks(thresholds[::2])
ax2_twin2.set_xticklabels([f'n={n}' for n in n_samples[::2]], fontsize=8, color='gray')
ax2_twin2.tick_params(axis='x', colors='gray')

plt.tight_layout()
plt.savefig('Fig04_Critical_Region_Curves.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig04_Critical_Region_Curves.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 4: Critical Region Curves (2 panels)")


# ============================================================
# ALTERNATIVE: THREE-PANEL VERSION (more detailed)
# ============================================================

fig2 = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, 3, wspace=0.28)

# ===== Panel (a): Coverage =====
ax1 = fig2.add_subplot(gs[0, 0])

ax1.axhline(90, color='black', linestyle='--', linewidth=2, alpha=0.5, label='90% Target')
ax1.fill_between(thresholds, 90-2, 90+2, alpha=0.1, color='gray', label='±2% Tolerance')

ax1.plot(thresholds, cp_picps, 'o-', color=CP_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='Standard CP')
ax1.plot(thresholds, pccp_picps, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='PCCP')

# Value labels at key points
for thresh in [50, 30, 15]:
    if thresh in thresholds:
        idx = thresholds.index(thresh)
        ax1.annotate(f'{cp_picps[idx]:.0f}%', (thresh, cp_picps[idx]), 
                     textcoords="offset points", xytext=(-12, 8), fontsize=9, color=CP_COLOR)
        ax1.annotate(f'{pccp_picps[idx]:.0f}%', (thresh, pccp_picps[idx]), 
                     textcoords="offset points", xytext=(12, -12), fontsize=9, color=PCCP_COLOR)

ax1.set_xlabel('RUL Threshold (cycles)', fontsize=11)
ax1.set_ylabel('Empirical Coverage (%)', fontsize=11)
ax1.set_title('(a) Coverage by RUL Region', fontsize=12, fontweight='bold')
ax1.set_xlim(max(thresholds) + 5, min(thresholds) - 2)
ax1.set_ylim(85, 105)
ax1.legend(loc='lower left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Theorem 1 box
ax1.text(0.97, 0.03, 'Theorem 1:\nCP ≈ PCCP\nat all regions', 
         transform=ax1.transAxes, fontsize=9, va='bottom', ha='right',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.4))

# ===== Panel (b): Interval Width =====
ax2 = fig2.add_subplot(gs[0, 1])

ax2.plot(thresholds, cp_mpiws, 'o-', color=CP_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='Standard CP')
ax2.plot(thresholds, pccp_mpiws, 's-', color=PCCP_COLOR, linewidth=2.5, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='PCCP')
ax2.fill_between(thresholds, pccp_mpiws, cp_mpiws, alpha=0.2, color=PCCP_COLOR,
                 label='Width Reduction')

# Improvement labels
for thresh in [50, 30, 15]:
    if thresh in thresholds:
        idx = thresholds.index(thresh)
        mid = (cp_mpiws[idx] + pccp_mpiws[idx]) / 2
        impr = mpiw_improvements[idx]
        ax2.annotate(f'-{impr:.0f}%', (thresh, mid), ha='center', fontsize=9, 
                     color=PCCP_COLOR, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax2.set_xlabel('RUL Threshold (cycles)', fontsize=11)
ax2.set_ylabel('Mean Prediction Interval Width (cycles)', fontsize=11)
ax2.set_title('(b) Interval Width by RUL Region', fontsize=12, fontweight='bold')
ax2.set_xlim(max(thresholds) + 5, min(thresholds) - 2)
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

# Corollary 1 box
ax2.text(0.03, 0.97, 'Corollary 1:\nPCCP width ≤ CP width\n(efficiency gain)', 
         transform=ax2.transAxes, fontsize=9, va='top', ha='left',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.4))

# ===== Panel (c): Physical Violations =====
ax3 = fig2.add_subplot(gs[0, 2])

# Fill area under CP curve
ax3.fill_between(thresholds, 0, cp_negs, alpha=0.3, color=INFEAS_COLOR)

ax3.plot(thresholds, cp_negs, 'o-', color=INFEAS_COLOR, linewidth=3, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='Standard CP')
ax3.plot(thresholds, [0]*len(thresholds), 's-', color=PCCP_COLOR, linewidth=3, markersize=10,
         markeredgecolor='white', markeredgewidth=2, label='PCCP (always 0%)')

# Value labels
for i, (thresh, neg) in enumerate(zip(thresholds, cp_negs)):
    if thresh in [125, 75, 50, 30, 15, 5]:
        offset_y = 6 if neg < 85 else -15
        ax3.annotate(f'{neg:.0f}%', (thresh, neg), textcoords="offset points", 
                     xytext=(0, offset_y), ha='center', fontsize=10, 
                     color=INFEAS_COLOR, fontweight='bold')

# Danger zone highlights
ax3.axhline(50, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
ax3.text(max(thresholds)-5, 52, '50% threshold', fontsize=8, color='orange')

ax3.set_xlabel('RUL Threshold (cycles)', fontsize=11)
ax3.set_ylabel('Infeasible Intervals (%)', fontsize=11)
ax3.set_title('(c) Physical Violations by RUL Region', fontsize=12, fontweight='bold')
ax3.set_xlim(max(thresholds) + 5, min(thresholds) - 2)
ax3.set_ylim(-5, 110)
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

# Key finding
ax3.annotate('Violations increase\ndramatically\nnear failure', 
             xy=(20, cp_negs[thresholds.index(20)] if 20 in thresholds else 70),
             xytext=(60, 75),
             fontsize=9, ha='center', color=INFEAS_COLOR,
             arrowprops=dict(arrowstyle='->', color=INFEAS_COLOR, lw=1.5),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=INFEAS_COLOR, alpha=0.9))

plt.tight_layout()
plt.savefig('Fig04_Critical_Region_3Panels.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig04_Critical_Region_3Panels.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 4: Critical Region (3 panels)")


# ============================================================
# PRINT SUMMARY TABLE
# ============================================================
print("\n" + "="*90)
print("📊 CRITICAL REGION ANALYSIS SUMMARY")
print("="*90)
print(f"{'Threshold':<12} {'Samples':<10} {'CP PICP':<12} {'PCCP PICP':<12} {'CP MPIW':<12} {'PCCP MPIW':<12} {'CP Neg%':<10}")
print("-"*90)
for r in critical_results_detailed:
    print(f"RUL<{r['threshold']:<7} {r['n']:<10} {r['CP_PICP']:.1f}%{'':<7} {r['PCCP_PICP']:.1f}%{'':<7} "
          f"{r['CP_MPIW']:.2f}{'':<7} {r['PCCP_MPIW']:.2f}{'':<7} {r['CP_Neg']:.1f}%")
print("="*90)
print("\n🔑 KEY FINDING: CP violations increase from ~15% (all data) to nearly 100% (RUL<5)")
print("   PCCP maintains 0% violations across ALL thresholds while preserving coverage.")

### Figure 9: Sensitivity Analysis

In [ ]:
# ============================================================
# FIGURE 9: SENSITIVITY ANALYSIS
# ============================================================

alpha_values = [0.20, 0.15, 0.10, 0.05, 0.02]
sensitivity = []

for alpha in alpha_values:
    q, _ = calibrate(model, X_cal, y_cal, alpha)
    res_cp = predict_intervals(model, X_test, q, apply_physics=False)
    res_pccp = predict_intervals(model, X_test, q, apply_physics=True)
    m_cp = compute_metrics(y_test, res_cp, 'CP')
    m_pccp = compute_metrics(y_test, res_pccp, 'PCCP')
    sensitivity.append({
        'nominal': 100 * (1 - alpha),
        'CP_PICP': m_cp['PICP'], 'PCCP_PICP': m_pccp['PICP'],
        'CP_MPIW': m_cp['MPIW'], 'PCCP_MPIW': m_pccp['MPIW'],
        'Neg%': m_cp['Neg%']
    })

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

nominals = [s['nominal'] for s in sensitivity]
cp_picps = [s['CP_PICP'] for s in sensitivity]
pccp_picps = [s['PCCP_PICP'] for s in sensitivity]
cp_mpiws = [s['CP_MPIW'] for s in sensitivity]
pccp_mpiws = [s['PCCP_MPIW'] for s in sensitivity]
neg_pcts = [s['Neg%'] for s in sensitivity]

# Panel (a): Calibration
ax1 = axes[0]
ax1.fill_between(nominals, np.array(nominals) - 2, np.array(nominals) + 2, alpha=0.15, color='gray', label='±2% band')
ax1.plot(nominals, nominals, 'k--', linewidth=2, label='Ideal')
ax1.plot(nominals, cp_picps, 'o-', color=COLORS['cp'], linewidth=2.5, markersize=12, label='CP')
ax1.plot(nominals, pccp_picps, 's-', color=COLORS['pccp'], linewidth=2.5, markersize=12, label='PCCP')
ax1.set_xlabel('Nominal Coverage (%)', fontsize=12)
ax1.set_ylabel('Empirical Coverage (%)', fontsize=12)
ax1.set_title('(a) Coverage Calibration', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right')
ax1.set_xlim(78, 100)
ax1.set_ylim(78, 100)
ax1.grid(True, alpha=0.3)

# Panel (b): Width
ax2 = axes[1]
ax2.plot(nominals, cp_mpiws, 'o-', color=COLORS['cp'], linewidth=2.5, markersize=12, label='CP')
ax2.plot(nominals, pccp_mpiws, 's-', color=COLORS['pccp'], linewidth=2.5, markersize=12, label='PCCP')
ax2.fill_between(nominals, pccp_mpiws, cp_mpiws, alpha=0.2, color=COLORS['pccp'])
for nom, cp_m, pccp_m in zip(nominals, cp_mpiws, pccp_mpiws):
    impr = 100 * (cp_m - pccp_m) / cp_m
    ax2.text(nom, (cp_m + pccp_m)/2, f'{impr:.0f}%', ha='center', fontsize=9, fontweight='bold', color=COLORS['pccp'])
ax2.set_xlabel('Nominal Coverage (%)', fontsize=12)
ax2.set_ylabel('MPIW (cycles)', fontsize=12)
ax2.set_title('(b) Interval Width', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Panel (c): Negative %
ax3 = axes[2]
ax3.bar(nominals, neg_pcts, width=3, color=COLORS['infeasible'], alpha=0.85, edgecolor='white')
ax3.set_xlabel('Nominal Coverage (%)', fontsize=12)
ax3.set_ylabel('Negative Intervals (%)', fontsize=12)
ax3.set_title('(c) CP Physical Violations', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')
for nom, neg in zip(nominals, neg_pcts):
    ax3.text(nom, neg + 0.5, f'{neg:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Figure 9: Sensitivity to Coverage Level', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig09_Sensitivity.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig09_Sensitivity.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 9: Sensitivity Analysis")

### Figure 10: Residual Analysis

In [ ]:
# ============================================================
# FIGURE 10: RESIDUAL ANALYSIS
# ============================================================

residuals = y_test - results_pccp_90['pred']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel (a): Residual histogram
ax1 = axes[0, 0]
ax1.hist(residuals, bins=60, color=COLORS['cp'], alpha=0.7, edgecolor='white', density=True)
# Fit normal distribution
mu, std = residuals.mean(), residuals.std()
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
ax1.plot(x_norm, stats.norm.pdf(x_norm, mu, std), color=COLORS['infeasible'], linewidth=2.5, 
         label=f'Normal($\mu$={mu:.1f}, $\sigma$={std:.1f})')
ax1.axvline(0, color='black', linestyle='--', linewidth=2)
ax1.set_xlabel('Residual (True - Predicted)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('(a) Residual Distribution', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Panel (b): Residuals vs Predicted
ax2 = axes[0, 1]
ax2.scatter(results_pccp_90['pred'], residuals, alpha=0.3, s=10, color=COLORS['cp'])
ax2.axhline(0, color=COLORS['infeasible'], linestyle='--', linewidth=2)
ax2.axhline(q_90, color=COLORS['neutral'], linestyle=':', linewidth=1.5, label=f'+q = {q_90:.1f}')
ax2.axhline(-q_90, color=COLORS['neutral'], linestyle=':', linewidth=1.5, label=f'-q = -{q_90:.1f}')
ax2.set_xlabel('Predicted RUL', fontsize=12)
ax2.set_ylabel('Residual', fontsize=12)
ax2.set_title('(b) Residuals vs Prediction', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Panel (c): Q-Q plot
ax3 = axes[1, 0]
stats.probplot(residuals, dist="norm", plot=ax3)
ax3.get_lines()[0].set_markerfacecolor(COLORS['cp'])
ax3.get_lines()[0].set_markeredgecolor(COLORS['cp'])
ax3.get_lines()[0].set_alpha(0.5)
ax3.get_lines()[1].set_color(COLORS['infeasible'])
ax3.get_lines()[1].set_linewidth(2)
ax3.set_title('(c) Q-Q Plot', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Panel (d): Residuals by RUL region
ax4 = axes[1, 1]
regions_data = {
    'RUL>50': residuals[y_test > 50],
    '30<RUL≤50': residuals[(y_test > 30) & (y_test <= 50)],
    '15<RUL≤30': residuals[(y_test > 15) & (y_test <= 30)],
    'RUL≤15': residuals[y_test <= 15]
}
bp = ax4.boxplot(regions_data.values(), labels=regions_data.keys(), patch_artist=True)
colors_box = [COLORS['pccp'], COLORS['cp'], COLORS['warning'], COLORS['infeasible']]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax4.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax4.set_ylabel('Residual', fontsize=12)
ax4.set_title('(d) Residuals by Region', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('Figure 10: Residual Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig10_Residuals.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig10_Residuals.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 10: Residual Analysis")

### Figure 11: Calibration Plot

In [ ]:
# ============================================================
# FIGURE 11: CALIBRATION PLOT (Predicted vs Actual)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel (a): Scatter plot
ax1 = axes[0]
ax1.scatter(y_test, results_pccp_90['pred'], alpha=0.3, s=15, color=COLORS['cp'])
ax1.plot([0, MAX_RUL], [0, MAX_RUL], 'r--', linewidth=2.5, label='Perfect Prediction')
ax1.fill_between([0, MAX_RUL], [-q_90, MAX_RUL - q_90], [q_90, MAX_RUL + q_90], 
                 alpha=0.15, color=COLORS['pccp'], label=f'90% Interval (±{q_90:.0f})')
ax1.set_xlabel('True RUL (cycles)', fontsize=12)
ax1.set_ylabel('Predicted RUL (cycles)', fontsize=12)
ax1.set_title('(a) Predicted vs Actual', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xlim(0, MAX_RUL + 10)
ax1.set_ylim(0, MAX_RUL + 10)
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# Panel (b): Hexbin density
ax2 = axes[1]
hb = ax2.hexbin(y_test, results_pccp_90['pred'], gridsize=30, cmap='YlGnBu', mincnt=1)
ax2.plot([0, MAX_RUL], [0, MAX_RUL], 'r--', linewidth=2.5)
ax2.set_xlabel('True RUL (cycles)', fontsize=12)
ax2.set_ylabel('Predicted RUL (cycles)', fontsize=12)
ax2.set_title('(b) Prediction Density', fontsize=13, fontweight='bold')
ax2.set_xlim(0, MAX_RUL + 10)
ax2.set_ylim(0, MAX_RUL + 10)
ax2.set_aspect('equal')
cb = plt.colorbar(hb, ax=ax2)
cb.set_label('Count', fontsize=11)

plt.suptitle('Figure 11: Model Calibration', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig11_Calibration.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig11_Calibration.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 11: Calibration Plot")

### Figure 12: Interval Width Distribution

In [ ]:
# ============================================================
# FIGURE 12: INTERVAL WIDTH DISTRIBUTION
# ============================================================

widths_cp = results_cp_90['upper'] - results_cp_90['lower']
widths_pccp = results_pccp_90['upper'] - results_pccp_90['lower']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel (a): Histograms - use shared bins to handle varying ranges
ax1 = axes[0]
# Create shared bins for both histograms
all_widths = np.concatenate([widths_cp, widths_pccp])
bins_shared = np.linspace(all_widths.min() - 1, all_widths.max() + 1, 40)

ax1.hist(widths_cp, bins=bins_shared, alpha=0.6, color=COLORS['cp'], 
         label=f'CP (mean={widths_cp.mean():.1f})', edgecolor='white')
ax1.hist(widths_pccp, bins=bins_shared, alpha=0.6, color=COLORS['pccp'], 
         label=f'PCCP (mean={widths_pccp.mean():.1f})', edgecolor='white')
ax1.axvline(widths_cp.mean(), color=COLORS['cp'], linestyle='--', linewidth=2)
ax1.axvline(widths_pccp.mean(), color=COLORS['pccp'], linestyle='--', linewidth=2)
ax1.set_xlabel('Interval Width (cycles)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('(a) Width Distribution', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Panel (b): Bar chart comparison
ax2 = axes[1]
stats_data = {
    'CP': [widths_cp.min(), widths_cp.mean(), widths_cp.max()],
    'PCCP': [widths_pccp.min(), widths_pccp.mean(), widths_pccp.max()]
}
x = np.arange(3)
width = 0.35
bars1 = ax2.bar(x - width/2, stats_data['CP'], width, label='CP', color=COLORS['cp'], alpha=0.8)
bars2 = ax2.bar(x + width/2, stats_data['PCCP'], width, label='PCCP', color=COLORS['pccp'], alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(['Min', 'Mean', 'Max'])
ax2.set_ylabel('Interval Width (cycles)', fontsize=12)
ax2.set_title('(b) Width Statistics', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9)

# Panel (c): Width vs True RUL
ax3 = axes[2]
# Sample for clearer visualization
sample_idx = np.random.choice(len(y_test), min(2000, len(y_test)), replace=False)
ax3.scatter(y_test[sample_idx], widths_cp[sample_idx], alpha=0.4, s=15, color=COLORS['cp'], label='CP')
ax3.scatter(y_test[sample_idx], widths_pccp[sample_idx], alpha=0.4, s=15, color=COLORS['pccp'], label='PCCP')

# Mean lines
ax3.axhline(widths_cp.mean(), color=COLORS['cp'], linewidth=2, linestyle='--', alpha=0.8)
ax3.axhline(widths_pccp.mean(), color=COLORS['pccp'], linewidth=2, linestyle='--', alpha=0.8)

ax3.set_xlabel('True RUL (cycles)', fontsize=12)
ax3.set_ylabel('Interval Width (cycles)', fontsize=12)
ax3.set_title('(c) Width vs True RUL', fontsize=13, fontweight='bold')
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

# Add annotation for width reduction
reduction = 100 * (widths_cp.mean() - widths_pccp.mean()) / widths_cp.mean()
ax3.text(0.95, 0.05, f'PCCP reduces width\nby {reduction:.1f}% on average', 
         transform=ax3.transAxes, fontsize=10, ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLORS['pccp'], alpha=0.9))

plt.suptitle('Figure 12: Interval Width Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig12_Width_Distribution.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig12_Width_Distribution.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 12: Interval Width Analysis")

### Figure 13: Physical Violations Heatmap

In [ ]:
# ============================================================
# FIGURE 13: PHYSICAL VIOLATIONS HEATMAP
# ============================================================

# Create bins for RUL
rul_bins = [0, 15, 30, 50, 75, 100, 125]
rul_labels = ['0-15', '15-30', '30-50', '50-75', '75-100', '100-125']

# Calculate metrics for each bin
heatmap_data = {'Coverage CP': [], 'Coverage PCCP': [], 'Neg% CP': [], 'Width Reduction': []}

for i in range(len(rul_bins) - 1):
    mask = (y_test >= rul_bins[i]) & (y_test < rul_bins[i+1])
    if mask.sum() > 0:
        # CP coverage
        cp_cov = 100 * ((y_test[mask] >= results_cp_90['lower'][mask]) & 
                        (y_test[mask] <= results_cp_90['upper'][mask])).mean()
        # PCCP coverage
        pccp_cov = 100 * ((y_test[mask] >= results_pccp_90['lower'][mask]) & 
                          (y_test[mask] <= results_pccp_90['upper'][mask])).mean()
        # Negative %
        neg_pct = 100 * np.mean(results_cp_90['lower_raw'][mask] < 0)
        # Width reduction
        cp_w = np.mean(results_cp_90['upper'][mask] - results_cp_90['lower'][mask])
        pccp_w = np.mean(results_pccp_90['upper'][mask] - results_pccp_90['lower'][mask])
        width_red = 100 * (cp_w - pccp_w) / cp_w if cp_w > 0 else 0
    else:
        cp_cov = pccp_cov = neg_pct = width_red = 0
    
    heatmap_data['Coverage CP'].append(cp_cov)
    heatmap_data['Coverage PCCP'].append(pccp_cov)
    heatmap_data['Neg% CP'].append(neg_pct)
    heatmap_data['Width Reduction'].append(width_red)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel (a): Negative % heatmap
ax1 = axes[0]
neg_matrix = np.array(heatmap_data['Neg% CP']).reshape(1, -1)
im1 = ax1.imshow(neg_matrix, cmap='Reds', aspect='auto', vmin=0, vmax=100)
ax1.set_xticks(range(len(rul_labels)))
ax1.set_xticklabels(rul_labels, fontsize=11)
ax1.set_yticks([0])
ax1.set_yticklabels(['CP'], fontsize=12)
ax1.set_xlabel('RUL Range (cycles)', fontsize=12)
ax1.set_title('(a) Physical Violations by Region', fontsize=13, fontweight='bold')

for j, val in enumerate(heatmap_data['Neg% CP']):
    text_color = 'white' if val > 50 else 'black'
    ax1.text(j, 0, f'{val:.0f}%', ha='center', va='center', fontsize=12, fontweight='bold', color=text_color)

cb1 = plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.15, shrink=0.8)
cb1.set_label('Negative Intervals (%)', fontsize=11)

# Panel (b): Width reduction
ax2 = axes[1]
wr_matrix = np.array(heatmap_data['Width Reduction']).reshape(1, -1)
im2 = ax2.imshow(wr_matrix, cmap='Greens', aspect='auto', vmin=0)
ax2.set_xticks(range(len(rul_labels)))
ax2.set_xticklabels(rul_labels, fontsize=11)
ax2.set_yticks([0])
ax2.set_yticklabels(['PCCP'], fontsize=12)
ax2.set_xlabel('RUL Range (cycles)', fontsize=12)
ax2.set_title('(b) Width Reduction (PCCP vs CP)', fontsize=13, fontweight='bold')

for j, val in enumerate(heatmap_data['Width Reduction']):
    text_color = 'white' if val > 30 else 'black'
    ax2.text(j, 0, f'{val:.0f}%', ha='center', va='center', fontsize=12, fontweight='bold', color=text_color)

cb2 = plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.15, shrink=0.8)
cb2.set_label('Width Reduction (%)', fontsize=11)

plt.suptitle('Figure 13: Regional Analysis Heatmaps', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig13_Heatmap.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig13_Heatmap.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 13: Regional Analysis Heatmaps")

### Figure 14: Summary Infographic

In [ ]:
# ============================================================
# FIGURE 14: SUMMARY INFOGRAPHIC
# ============================================================

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 4, height_ratios=[1, 1.2, 1], hspace=0.4, wspace=0.3)

# Title
fig.suptitle('PCCP: Physics-Constrained Conformal Prediction for RUL', 
             fontsize=18, fontweight='bold', y=0.98)

# Row 1: Key metrics cards
metrics_cards = [
    ('Coverage', f'{metrics_pccp_90["PICP"]:.1f}%', f'Target: 90%', COLORS['pccp']),
    ('MPIW', f'{metrics_pccp_90["MPIW"]:.1f}', f'-{mpiw_impr_90:.0f}% vs CP', COLORS['cp']),
    ('Physical', '100%', 'Consistency', COLORS['pccp']),
    ('RMSE', f'{metrics_pccp_90["RMSE"]:.1f}', 'cycles', COLORS['purple'])
]

for i, (title, value, subtitle, color) in enumerate(metrics_cards):
    ax = fig.add_subplot(gs[0, i])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    # Background card
    rect = FancyBboxPatch((0.05, 0.1), 0.9, 0.8, boxstyle="round,pad=0.02,rounding_size=0.1",
                          facecolor=color, alpha=0.2, edgecolor=color, linewidth=3)
    ax.add_patch(rect)
    
    ax.text(0.5, 0.75, title, ha='center', va='center', fontsize=13, fontweight='bold', color=color)
    ax.text(0.5, 0.45, value, ha='center', va='center', fontsize=24, fontweight='bold', color=COLORS['dark'])
    ax.text(0.5, 0.2, subtitle, ha='center', va='center', fontsize=10, color=COLORS['neutral'])

# Row 2: Main comparison chart
ax_main = fig.add_subplot(gs[1, :2])
x = np.arange(3)
width = 0.35
cp_vals = [metrics_cp_90['PICP'], metrics_cp_90['Phys%'], 100 - mpiw_impr_90]
pccp_vals = [metrics_pccp_90['PICP'], 100, 100]
labels = ['Coverage (%)', 'Physical (%)','Efficiency (%)']

bars1 = ax_main.bar(x - width/2, cp_vals, width, label='CP', color=COLORS['cp'], alpha=0.8)
bars2 = ax_main.bar(x + width/2, pccp_vals, width, label='PCCP', color=COLORS['pccp'], alpha=0.8)
ax_main.set_xticks(x)
ax_main.set_xticklabels(labels, fontsize=12)
ax_main.set_ylabel('Percentage (%)', fontsize=12)
ax_main.set_title('CP vs PCCP Comparison', fontsize=14, fontweight='bold')
ax_main.legend(loc='lower right', fontsize=11)
ax_main.set_ylim(0, 115)
ax_main.grid(True, alpha=0.3, axis='y')

# Row 2: Critical region
ax_crit = fig.add_subplot(gs[1, 2:])
regions = ['All', '<50', '<30', '<15']
neg_vals = [metrics_cp_90['Neg%']] + [critical_results[t]['CP_Neg%'] for t in critical_thresholds]
colors_crit = [COLORS['warning'] if v < 50 else COLORS['infeasible'] for v in neg_vals]
bars = ax_crit.bar(regions, neg_vals, color=colors_crit, alpha=0.85, edgecolor='white')
for bar, v in zip(bars, neg_vals):
    ax_crit.text(bar.get_x() + bar.get_width()/2, v + 2, f'{v:.0f}%', ha='center', fontsize=11, fontweight='bold')
ax_crit.set_xlabel('RUL Region', fontsize=12)
ax_crit.set_ylabel('CP Negative Intervals (%)', fontsize=12)
ax_crit.set_title('Physical Violations (PCCP = 0% always)', fontsize=14, fontweight='bold')
ax_crit.grid(True, alpha=0.3, axis='y')

# Row 3: Key takeaways
ax_text = fig.add_subplot(gs[2, :])
ax_text.axis('off')

takeaways = [
    "✓ Theorem 1 Validated: Coverage preserved (CP ≈ PCCP)",
    f"✓ Corollary 1 Validated: MPIW improved by {mpiw_impr_90:.1f}%",
    "✓ 100% Physical Consistency: All predictions are non-negative",
    "✓ Critical Region: CP fails up to 100% at RUL<15, PCCP always 0%"
]

for i, text in enumerate(takeaways):
    ax_text.text(0.5, 0.85 - i*0.25, text, ha='center', va='center', fontsize=13, 
                 fontweight='bold' if i == 0 else 'normal',
                 color=COLORS['pccp'] if '✓' in text else 'black',
                 transform=ax_text.transAxes)

plt.savefig('Fig14_Summary.png', dpi=600, facecolor='white', edgecolor='none')
plt.savefig('Fig14_Summary.pdf', dpi=600, facecolor='white', edgecolor='none')
plt.show()
print("✓ Figure 14: Summary Infographic")

---
## Save Results

In [ ]:
# Save all results
print("\n" + "="*70)
print("📊 PUBLICATION FIGURES - v8 SETTINGS")
print("="*70)
print("\nKey difference from old settings:")
print("  OLD: physics_loss = MSE + λ·non_negativity_penalty (λ=0.1)")
print("  V8:  pure MSE loss (no physics penalty during training)")
print("  V8:  5 UQ methods (CP, MC Dropout, Ensemble, QR, CQR)")
print("\nAll figures use v8 trained models and results.")
